# 08 — Binary Damage Detection

## Purpose

This notebook contains the Paderborn **binary-classification** experiments. It is not the missing Stage 2 bearing-level cross-validation notebook. It studies whether vibration measurements indicate a Healthy or Damaged bearing through three related tasks:

1. **Healthy vs Outer:** fault classes 0 (Healthy) and 1 (Outer).
2. **Healthy vs Inner:** fault classes 0 (Healthy) and 2 (Inner).
3. **Integrated Healthy vs Damaged:** class 0 maps to Healthy; classes 1 (Outer) and 2 (Inner) map to Damaged. Class 3 (Both) is explicitly excluded.

Each task has its own bearing-level split definition. Do not combine their train/validation/test counts as if they were one split.

## Inputs and execution order

- Run notebooks 01–03 first.
- Notebook 03 must have created `data/processed/paderborn_vibration_features.csv`.
- Raw Paderborn MATLAB recordings must be available below `data/raw/`, because this notebook reopens files to calculate BPFO/BPFI envelope-energy features.
- Start JupyterLab with its working directory set to the `notebooks` folder. Project paths in this repository copy are relative to the repository, not a specific Windows user directory.
- The core notebook environment includes NumPy, Pandas, SciPy, Matplotlib and scikit-learn. TensorFlow is not required.

## Feature representation

The notebook starts from 22 general statistical/spectral features, then calculates six BPFO-related and six BPFI-related envelope-energy features. The combined representation contains 34 features (22 + 6 + 6).

## Important interpretation limits

- The integrated binary task excludes the Both class. It does not solve four-class Healthy/Outer/Inner/Both classification.
- The integrated test set contains 320 recordings from four physical bearings. Repeated recordings from one bearing are related observations, not independent specimens.
- The final test subset was inspected during iterative experimentation. The reported metrics are project results for this selected split, not a pristine untouched external test or proof of universal generalisation.
- Notebook 09 packages the separate 34-feature inference baseline using development/test feature tables exported by this notebook.

The original saved outputs have been removed from this repository copy to keep it portable. Running from top to bottom regenerates the tables and plots when the required local data files are present.

In [ ]:
import os
import pandas as pd

from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name.lower() == "notebooks"
    else Path.cwd()
)
BASE_DIR = str(PROJECT_ROOT)
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

FEATURE_FILE = os.path.join(
    PROCESSED_DIR,
    "paderborn_vibration_features.csv"
)

df = pd.read_csv(FEATURE_FILE)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Fault class distribution:")
print(df["fault_class"].value_counts())

print("\nBinary label distribution:")
print(df["binary_label"].value_counts())

print("\nFault class vs binary label:")
print(pd.crosstab(df["fault_class"], df["binary_label"]))

# 1. Load the feature table and inspect binary labels

Read the processed feature table created by Notebook 03, inspect the original four-class labels, and define the binary subsets.

In [ ]:
df_binary = df[df["fault_class"].isin([0, 1, 2])].copy()

print("Binary experiment dataset shape:", df_binary.shape)

print("\nFault classes:")
print(df_binary["fault_class"].value_counts().sort_index())

print("\nBinary labels:")
print(df_binary["binary_label"].value_counts().sort_index())

In [ ]:
df_healthy_outer = df[df["fault_class"].isin([0, 1])].copy()

print("Healthy vs Outer dataset shape:", df_healthy_outer.shape)

print("\nClass distribution:")
print(df_healthy_outer["fault_class"].value_counts().sort_index())

print("\nBinary labels:")
print(df_healthy_outer["binary_label"].value_counts().sort_index())

print("\nBearings:")
print(df_healthy_outer["bearing"].nunique())
print(sorted(df_healthy_outer["bearing"].unique()))

# 2. Define bearing-level splits and the 22 general features

The Healthy-vs-Outer experiment defines its own training, validation and test bearing lists. The Healthy-vs-Inner and integrated experiments define separate lists later in this notebook.

In [ ]:
TRAIN_BEARINGS = [
    "K001", "K002", "K003", "K004",
    "KA01", "KA04", "KA05", "KA06", "KA07",
    "KA09", "KA15", "KA16", "KA30"
]

VAL_BEARINGS = [
    "K005", "KA03", "KA08"
]

TEST_BEARINGS = [
    "K006", "KA22"
]

def assign_split(bearing):
    if bearing in TRAIN_BEARINGS:
        return "train"
    elif bearing in VAL_BEARINGS:
        return "validation"
    elif bearing in TEST_BEARINGS:
        return "test"
    return "excluded"

df_healthy_outer["split"] = df_healthy_outer["bearing"].apply(assign_split)

print("Split distribution:")
print(df_healthy_outer["split"].value_counts())

print("\nClass distribution by split:")
print(
    pd.crosstab(
        df_healthy_outer["split"],
        df_healthy_outer["fault_class"]
    )
)

print("\nBearings by split:")
for split in ["train", "validation", "test"]:
    print(f"\n{split.upper()}:")
    print(
        sorted(
            df_healthy_outer.loc[
                df_healthy_outer["split"] == split, "bearing"
            ].unique()
        )
    )

In [ ]:
FEATURES_22 = [
    "mean",
    "std",
    "min",
    "max",
    "kurtosis",
    "skewness",
    "crest_factor",
    "dominant_frequency",
    "dominant_magnitude",
    "spectral_centroid",
    "spectral_bandwidth",
    "band_power_0_500",
    "band_power_500_1000",
    "band_power_1000_2000",
    "band_power_2000_3000",
    "band_power_3000_4000",
    "band_power_4000_5000",
    "band_power_5000_6000",
    "band_power_6000_7000",
    "band_power_7000_8000",
    "band_power_8000_9000",
    "band_power_9000_10000"
]

train_df = df_healthy_outer[df_healthy_outer["split"] == "train"].copy()
val_df = df_healthy_outer[df_healthy_outer["split"] == "validation"].copy()
test_df = df_healthy_outer[df_healthy_outer["split"] == "test"].copy()

X_train = train_df[FEATURES_22]
y_train = train_df["binary_label"]

X_val = val_df[FEATURES_22]
y_val = val_df["binary_label"]

X_test = test_df[FEATURES_22]
y_test = test_df["binary_label"]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# 3. Examine feature distributions and outliers

Inspect feature ranges, outlier counts and differences across Healthy/Outer training recordings, bearings and operating conditions.

In [ ]:
import numpy as np
import pandas as pd

outlier_summary = []

for feature in FEATURES_22:
    values = train_df[feature].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((values < lower) | (values > upper)).sum()

    outlier_summary.append({
        "feature": feature,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": count,
        "outlier_percent": 100 * count / len(values)
    })

outlier_summary = pd.DataFrame(outlier_summary)

print(outlier_summary.sort_values(
    "outlier_percent", ascending=False
).to_string(index=False))

In [ ]:
for class_label, class_name in [(0, "Healthy"), (1, "Outer")]:
    class_data = train_df[train_df["fault_class"] == class_label]

    print(f"\n{'=' * 20} {class_name} {'=' * 20}")

    rows = []

    for feature in FEATURES_22:
        values = class_data[feature].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = ((values < lower) | (values > upper)).sum()

        rows.append({
            "feature": feature,
            "outlier_count": count,
            "outlier_percent": 100 * count / len(values)
        })

    class_summary = pd.DataFrame(rows)

    print(
        class_summary.sort_values(
            "outlier_percent",
            ascending=False
        ).to_string(index=False)
    )

In [ ]:
class_outlier_tables = {}

for class_label, class_name in [(0, "Healthy"), (1, "Outer")]:
    class_data = train_df[train_df["fault_class"] == class_label]
    rows = []

    for feature in FEATURES_22:
        values = class_data[feature].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = ((values < lower) | (values > upper)).sum()

        rows.append({
            "feature": feature,
            "outlier_percent": 100 * count / len(values)
        })

    class_outlier_tables[class_name] = pd.DataFrame(rows)

outlier_compare = (
    class_outlier_tables["Healthy"]
    .rename(columns={"outlier_percent": "healthy_outlier_%"})
    .merge(
        class_outlier_tables["Outer"]
        .rename(columns={"outlier_percent": "outer_outlier_%"}),
        on="feature"
    )
)

outlier_compare["max_outlier_%"] = outlier_compare[
    ["healthy_outlier_%", "outer_outlier_%"]
].max(axis=1)

print(
    outlier_compare
    .sort_values("max_outlier_%", ascending=False)
    .to_string(index=False)
)

In [ ]:
suspect_features = [
    "skewness",
    "kurtosis",
    "band_power_0_500",
    "band_power_500_1000",
    "band_power_2000_3000",
    "band_power_9000_10000",
    "dominant_magnitude"
]

for feature in suspect_features:
    print(f"\n{'=' * 15} {feature} {'=' * 15}")

    values = train_df[feature]
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_rows = train_df[
        (train_df[feature] < lower) |
        (train_df[feature] > upper)
    ]

    print("Outlier count:", len(outlier_rows))

    print("\nBy bearing:")
    print(
        outlier_rows["bearing"]
        .value_counts()
        .head(10)
    )

    print("\nBy operating condition:")
    print(
        outlier_rows["operating_condition"]
        .value_counts()
    )

print("\nMinimum band-power values:")
band_power_cols = [c for c in FEATURES_22 if c.startswith("band_power_")]
print(train_df[band_power_cols].min().sort_values())

In [ ]:
bearing_summary = (
    train_df
    .groupby(["fault_class", "bearing"])[FEATURES_22]
    .mean()
)

for feature in [
    "skewness",
    "kurtosis",
    "band_power_0_500",
    "band_power_500_1000",
    "band_power_2000_3000",
    "band_power_9000_10000",
    "dominant_magnitude"
]:
    print(f"\n{'=' * 15} {feature} {'=' * 15}")
    print(
        bearing_summary[feature]
        .unstack(0)
        .sort_index()
        .to_string()
    )

In [ ]:
representative_bearings = ["K001", "KA04", "KA16", "KA05"]

for bearing in representative_bearings:
    rows = train_df[train_df["bearing"] == bearing]

    print(f"\n{bearing}")
    print(
        rows[
            ["file", "operating_condition", "measurement_number", "fault_class"]
        ].head(8).to_string(index=False)
    )

In [ ]:
representative_bearings = ["K001", "KA04", "KA16", "KA05"]

for bearing in representative_bearings:
    rows = train_df[train_df["bearing"] == bearing]

    print(f"\n{bearing}")
    print(
        rows[
            ["file", "operating_condition", "measurement_number", "fault_class"]
        ].head(8).to_string(index=False)
    )

# 4. Inspect raw vibration, spectra and bearing characteristic frequencies

Reopen representative Paderborn MAT files, extract vibration_1 and speed, plot waveforms, calculate FFT/envelope spectra, and calculate shaft, BPFO, BPFI, FTF and BSF frequencies using the nominal 64 kHz rate and 6203 bearing geometry.

In [ ]:
import os
import numpy as np
import scipy.io as sio

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")

def find_mat_file(filename):
    for root, _, files in os.walk(RAW_DIR):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"{filename} not found under {RAW_DIR}")


def get_vibration_signal(mat_file):
    mat = sio.loadmat(
        mat_file,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]

    if len(data_keys) != 1:
        raise ValueError(f"Unexpected MATLAB structure in {mat_file}")

    data = mat[data_keys[0]][0, 0]
    y_channels = np.asarray(data.Y).flatten()

    for channel in y_channels:
        name = channel.Name

        if isinstance(name, np.ndarray):
            name = str(name.flatten()[0])

        if name == "vibration_1":
            signal = np.asarray(channel.Data).squeeze()
            return signal.astype(np.float64)

    raise ValueError(f"vibration_1 channel not found in {mat_file}")


representative_files = {
    "K001": "N09_M07_F10_K001_1.mat",
    "KA04": "N09_M07_F10_KA04_1.mat",
    "KA16": "N09_M07_F10_KA16_1.mat",
    "KA05": "N09_M07_F10_KA05_1.mat"
}

raw_signals = {}

for bearing, filename in representative_files.items():
    path = find_mat_file(filename)
    signal = get_vibration_signal(path)
    raw_signals[bearing] = signal

    print(f"\n{bearing}")
    print("File:", path)
    print("Samples:", len(signal))
    print("Min:", signal.min())
    print("Max:", signal.max())
    print("Mean:", signal.mean())
    print("Std:", signal.std())
    print("RMS:", np.sqrt(np.mean(signal ** 2)))

In [ ]:
import matplotlib.pyplot as plt

WINDOW_SECONDS = 0.05
N_SAMPLES = int(WINDOW_SECONDS * 64000)

for bearing, signal in raw_signals.items():
    segment = signal[:N_SAMPLES]
    time = np.arange(len(segment)) / 64000

    plt.figure(figsize=(12, 4))
    plt.plot(time, segment)
    plt.title(f"{bearing} - Raw Vibration Signal")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.fft import rfft, rfftfreq
import matplotlib.pyplot as plt
import numpy as np

FS = 64000
SEGMENT_SECONDS = 1
MAX_FREQ = 10000

for bearing, signal in raw_signals.items():

    # Use the first 1 second
    segment = signal[:FS].copy()

    # Remove DC
    segment = segment - np.mean(segment)

    # Hann window
    window = np.hanning(len(segment))
    windowed = segment * window

    # FFT
    spectrum = np.abs(rfft(windowed))
    frequencies = rfftfreq(len(windowed), d=1 / FS)

    # 0–10 kHz
    mask = frequencies <= MAX_FREQ

    plt.figure(figsize=(12, 4))
    plt.plot(frequencies[mask], spectrum[mask])

    plt.title(f"{bearing} - FFT Spectrum")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.signal import butter, sosfiltfilt, hilbert
from scipy.fft import rfft, rfftfreq
import numpy as np
import matplotlib.pyplot as plt

FS = 64000

# Initial high-frequency band used to isolate vibration resonances
LOWCUT = 2000
HIGHCUT = 10000

# Envelope spectrum shown up to 1000 Hz
MAX_ENVELOPE_FREQ = 1000


def envelope_spectrum(signal, fs, lowcut, highcut, max_freq):
    signal = signal[:fs].copy()

    # Remove DC
    signal = signal - np.mean(signal)

    # Band-pass filter
    sos = butter(
        4,
        [lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    filtered = sosfiltfilt(sos, signal)

    # Hilbert envelope
    analytic_signal = hilbert(filtered)
    envelope = np.abs(analytic_signal)

    # Remove envelope DC component
    envelope = envelope - np.mean(envelope)

    # Envelope FFT
    spectrum = np.abs(rfft(envelope))
    frequencies = rfftfreq(len(envelope), d=1 / fs)

    mask = frequencies <= max_freq

    return frequencies[mask], spectrum[mask]


plt.figure(figsize=(12, 6))

for bearing, signal in raw_signals.items():

    frequencies, spectrum = envelope_spectrum(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        MAX_ENVELOPE_FREQ
    )

    # Normalize so we compare frequency locations,
    # not the very different absolute amplitudes.
    spectrum = spectrum / (np.max(spectrum) + 1e-12)

    plt.plot(
        frequencies,
        spectrum,
        label=bearing
    )

plt.title("Normalized Envelope Spectra")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized Magnitude")
plt.xlim(0, MAX_ENVELOPE_FREQ)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the measured channels inside one representative .mat file

sample_file = find_mat_file("N09_M07_F10_KA04_1.mat")

mat = sio.loadmat(
    sample_file,
    squeeze_me=False,
    struct_as_record=False
)

data_keys = [key for key in mat.keys() if not key.startswith("__")]
data = mat[data_keys[0]][0, 0]

y_channels = np.asarray(data.Y).flatten()

print("Channels found in the .mat file:\n")

for channel in y_channels:
    name = channel.Name

    if isinstance(name, np.ndarray):
        name = str(name.flatten()[0])

    signal = np.asarray(channel.Data).squeeze()

    print(
        f"{name:25s} | "
        f"samples = {len(signal):7d} | "
        f"min = {np.min(signal):10.4f} | "
        f"max = {np.max(signal):10.4f}"
    )

In [ ]:
import numpy as np

# Bearing geometry: 6203
N_BALLS = 8
BALL_DIAMETER = 6.75   # mm
PITCH_DIAMETER = 29.05 # mm
PRESSURE_ANGLE_DEG = 0

cos_theta = np.cos(np.deg2rad(PRESSURE_ANGLE_DEG))


def extract_channel(data, channel_name):
    y_channels = np.asarray(data.Y).flatten()

    for channel in y_channels:
        name = channel.Name

        if isinstance(name, np.ndarray):
            name = str(name.flatten()[0])

        if name == channel_name:
            return np.asarray(channel.Data).squeeze().astype(np.float64)

    raise ValueError(f"{channel_name} not found")


def get_mean_speed(filename):
    path = find_mat_file(filename)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]
    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(data, "speed")

    return np.mean(speed)


def calculate_fault_frequencies(rpm):
    fr = rpm / 60.0

    bpfo = (
        N_BALLS / 2
        * fr
        * (1 - (BALL_DIAMETER / PITCH_DIAMETER) * cos_theta)
    )

    bpfi = (
        N_BALLS / 2
        * fr
        * (1 + (BALL_DIAMETER / PITCH_DIAMETER) * cos_theta)
    )

    ftf = (
        0.5
        * fr
        * (1 - (BALL_DIAMETER / PITCH_DIAMETER) * cos_theta)
    )

    bsf = (
        PITCH_DIAMETER / (2 * BALL_DIAMETER)
        * fr
        * (
            1
            - (BALL_DIAMETER / PITCH_DIAMETER) ** 2
            * cos_theta ** 2
        )
    )

    return fr, bpfo, bpfi, ftf, bsf


for bearing, filename in representative_files.items():

    rpm = get_mean_speed(filename)

    fr, bpfo, bpfi, ftf, bsf = calculate_fault_frequencies(rpm)

    print(f"\n{bearing}")
    print(f"Measured speed : {rpm:.2f} rpm")
    print(f"Shaft frequency: {fr:.2f} Hz")
    print(f"BPFO           : {bpfo:.2f} Hz")
    print(f"BPFI           : {bpfi:.2f} Hz")
    print(f"FTF            : {ftf:.2f} Hz")
    print(f"BSF            : {bsf:.2f} Hz")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use the calculated BPFO/BPFI at approximately 900 rpm
rpm_reference = 900.0
fr, bpfo, bpfi, ftf, bsf = calculate_fault_frequencies(rpm_reference)

print(f"Expected BPFO = {bpfo:.2f} Hz")
print(f"Expected BPFI = {bpfi:.2f} Hz")

plt.figure(figsize=(14, 6))

for bearing, signal in raw_signals.items():

    frequencies, spectrum = envelope_spectrum(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        MAX_ENVELOPE_FREQ
    )

    spectrum = spectrum / (np.max(spectrum) + 1e-12)

    plt.plot(
        frequencies,
        spectrum,
        label=bearing
    )

# BPFO harmonics
for harmonic in range(1, 6):
    freq = harmonic * bpfo

    plt.axvline(
        freq,
        linestyle="--",
        alpha=0.5
    )

    plt.text(
        freq,
        1.02,
        f"{harmonic}×BPFO",
        rotation=90,
        ha="center",
        va="bottom"
    )

plt.xlim(0, 300)
plt.ylim(0, 1.1)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized Envelope Magnitude")
plt.title("Envelope Spectrum with Expected BPFO Harmonics")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
analysis_bearings = [
    "K001", "K002", "K003", "K004",
    "KA01", "KA04", "KA05", "KA06",
    "KA07", "KA09", "KA15", "KA16", "KA30"
]

records = []

for bearing in analysis_bearings:

    row = train_df[
        (train_df["bearing"] == bearing) &
        (train_df["operating_condition"] == "N09_M07_F10") &
        (train_df["measurement_number"] == 1)
    ]

    if row.empty:
        print(f"Skipping {bearing}: measurement not found")
        continue

    filename = row.iloc[0]["file"]
    path = find_mat_file(filename)

    signal = get_vibration_signal(path)

    frequencies, spectrum = envelope_spectrum(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        300
    )

    # Expected BPFO for approximately 900 rpm
    _, bpfo, _, _, _ = calculate_fault_frequencies(900)

    # Look around the first 5 BPFO harmonics
    harmonic_ratios = []

    for harmonic in range(1, 6):

        target = harmonic * bpfo

        # ±3 Hz around expected frequency
        band = np.abs(frequencies - target) <= 3

        if np.any(band):
            peak = np.max(spectrum[band])
        else:
            peak = np.nan

        harmonic_ratios.append(peak)

    # Normalize by the strongest envelope component
    harmonic_ratios = np.array(harmonic_ratios)
    harmonic_ratios = harmonic_ratios / (np.max(spectrum) + 1e-12)

    records.append({
        "bearing": bearing,
        "fault_class": row.iloc[0]["fault_class"],
        "bpfo_1x": harmonic_ratios[0],
        "bpfo_2x": harmonic_ratios[1],
        "bpfo_3x": harmonic_ratios[2],
        "bpfo_4x": harmonic_ratios[3],
        "bpfo_5x": harmonic_ratios[4],
    })

bpfo_df = pd.DataFrame(records)

display(bpfo_df)

In [ ]:
# Step 6M-1
# Check BPFO stability across all 20 measurements
# for the training Healthy vs Outer bearings.

analysis_bearings = [
    "K001", "K002", "K003", "K004",
    "KA01", "KA04", "KA05", "KA06",
    "KA07", "KA09", "KA15", "KA16", "KA30"
]

all_bpfo_records = []

for bearing in analysis_bearings:

    bearing_rows = train_df[
        (train_df["bearing"] == bearing) &
        (train_df["operating_condition"] == "N09_M07_F10")
    ].sort_values("measurement_number")

    print(f"Processing {bearing}: {len(bearing_rows)} measurements")

    for _, row in bearing_rows.iterrows():

        filename = row["file"]
        path = find_mat_file(filename)

        signal = get_vibration_signal(path)

        # Actual measured speed for this recording
        mat = sio.loadmat(
            path,
            squeeze_me=False,
            struct_as_record=False
        )

        data_keys = [key for key in mat.keys() if not key.startswith("__")]
        data = mat[data_keys[0]][0, 0]

        speed = extract_channel(data, "speed")
        rpm = np.mean(speed)

        _, bpfo, _, _, _ = calculate_fault_frequencies(rpm)

        frequencies, spectrum = envelope_spectrum(
            signal,
            FS,
            LOWCUT,
            HIGHCUT,
            300
        )

        # Normalize by maximum envelope magnitude
        spectrum = spectrum / (np.max(spectrum) + 1e-12)

        harmonic_values = []

        for harmonic in range(1, 6):

            target = harmonic * bpfo

            band = np.abs(frequencies - target) <= 3

            if np.any(band):
                value = np.max(spectrum[band])
            else:
                value = np.nan

            harmonic_values.append(value)

        all_bpfo_records.append({
            "bearing": bearing,
            "fault_class": row["fault_class"],
            "measurement_number": row["measurement_number"],
            "rpm": rpm,
            "bpfo": bpfo,
            "bpfo_1x": harmonic_values[0],
            "bpfo_2x": harmonic_values[1],
            "bpfo_3x": harmonic_values[2],
            "bpfo_4x": harmonic_values[3],
            "bpfo_5x": harmonic_values[4]
        })

bpfo_all_df = pd.DataFrame(all_bpfo_records)

print("\nResult shape:", bpfo_all_df.shape)

display(bpfo_all_df.head())

In [ ]:
bpfo_summary = (
    bpfo_all_df
    .groupby(["bearing", "fault_class"])
    .agg(
        bpfo_1x_mean=("bpfo_1x", "mean"),
        bpfo_1x_std=("bpfo_1x", "std"),
        bpfo_2x_mean=("bpfo_2x", "mean"),
        bpfo_2x_std=("bpfo_2x", "std"),
        bpfo_3x_mean=("bpfo_3x", "mean"),
        bpfo_3x_std=("bpfo_3x", "std"),
        bpfo_4x_mean=("bpfo_4x", "mean"),
        bpfo_4x_std=("bpfo_4x", "std"),
        bpfo_5x_mean=("bpfo_5x", "mean"),
        bpfo_5x_std=("bpfo_5x", "std"),
    )
    .reset_index()
)

display(bpfo_summary.round(3))

In [ ]:
plt.figure(figsize=(14, 6))

for bearing in analysis_bearings:
    subset = bpfo_all_df[bpfo_all_df["bearing"] == bearing]

    plt.plot(
        subset["measurement_number"],
        subset["bpfo_1x"],
        marker="o",
        linewidth=1.5,
        label=bearing
    )

plt.xlabel("Measurement Number")
plt.ylabel("Normalized 1× BPFO magnitude")
plt.title("1× BPFO Stability Across 20 Measurements")
plt.grid(True)
plt.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
bpfo_features = [
    "bpfo_1x",
    "bpfo_2x",
    "bpfo_3x",
    "bpfo_4x",
    "bpfo_5x"
]

class_summary = (
    bpfo_all_df
    .groupby("fault_class")[bpfo_features]
    .agg(["mean", "std"])
)

display(class_summary.round(3))

In [ ]:
plot_df = (
    bpfo_all_df
    .groupby(["bearing", "fault_class"])[bpfo_features]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))

for _, row in plot_df.iterrows():

    label = (
        f"{row['bearing']} "
        f"({'Healthy' if row['fault_class'] == 0 else 'Outer'})"
    )

    plt.plot(
        range(1, 6),
        row[bpfo_features].values,
        marker="o",
        linewidth=2,
        label=label
    )

plt.xticks(
    range(1, 6),
    ["1× BPFO", "2× BPFO", "3× BPFO", "4× BPFO", "5× BPFO"]
)

plt.xlabel("BPFO Harmonic")
plt.ylabel("Mean Normalized Envelope Magnitude")
plt.title("Mean BPFO Harmonic Profiles Across Training Bearings")
plt.grid(True)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
def extract_bpfo_energy_features(
    signal,
    rpm,
    fs=64000,
    lowcut=2000,
    highcut=10000,
    max_envelope_freq=300,
    tolerance=3
):
    frequencies, spectrum = envelope_spectrum(
        signal,
        fs,
        lowcut,
        highcut,
        max_envelope_freq
    )

    # Spectral power
    spectral_power = spectrum ** 2

    # Total envelope-spectrum energy from 20–300 Hz
    total_band = (frequencies >= 20) & (frequencies <= max_envelope_freq)

    total_energy = np.sum(spectral_power[total_band])

    _, bpfo, _, _, _ = calculate_fault_frequencies(rpm)

    features = {}

    for harmonic in range(1, 6):

        target = harmonic * bpfo

        band = np.abs(frequencies - target) <= tolerance

        harmonic_energy = np.sum(spectral_power[band])

        features[f"bpfo_{harmonic}x_energy_ratio"] = (
            harmonic_energy / (total_energy + 1e-12)
        )

    return features


energy_records = []

for bearing in analysis_bearings:

    bearing_rows = train_df[
        (train_df["bearing"] == bearing) &
        (train_df["operating_condition"] == "N09_M07_F10")
    ].sort_values("measurement_number")

    for _, row in bearing_rows.iterrows():

        path = find_mat_file(row["file"])

        signal = get_vibration_signal(path)

        mat = sio.loadmat(
            path,
            squeeze_me=False,
            struct_as_record=False
        )

        data_keys = [key for key in mat.keys() if not key.startswith("__")]
        data = mat[data_keys[0]][0, 0]

        speed = extract_channel(data, "speed")
        rpm = np.mean(speed)

        features = extract_bpfo_energy_features(
            signal,
            rpm
        )

        features.update({
            "bearing": bearing,
            "fault_class": row["fault_class"],
            "measurement_number": row["measurement_number"]
        })

        energy_records.append(features)


bpfo_energy_df = pd.DataFrame(energy_records)

print("Shape:", bpfo_energy_df.shape)

display(bpfo_energy_df.head())

In [ ]:
energy_features = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio"
]

bpfo_energy_summary = (
    bpfo_energy_df
    .groupby(["bearing", "fault_class"])[energy_features]
    .agg(["mean", "std"])
)

display(bpfo_energy_summary.round(5))

In [ ]:
bpfo_energy_class_summary = (
    bpfo_energy_df
    .groupby("fault_class")[energy_features]
    .agg(["mean", "std"])
)

display(bpfo_energy_class_summary.round(5))

In [ ]:
# Step 6O
# Calculate BPFO energy-ratio features across ALL
# training bearings and ALL operating conditions.

bpfo_condition_records = []

for _, row in train_df.iterrows():

    path = find_mat_file(row["file"])

    signal = get_vibration_signal(path)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]
    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(data, "speed")
    rpm = np.mean(speed)

    features = extract_bpfo_energy_features(
        signal,
        rpm
    )

    features.update({
        "bearing": row["bearing"],
        "operating_condition": row["operating_condition"],
        "measurement_number": row["measurement_number"],
        "fault_class": row["fault_class"]
    })

    bpfo_condition_records.append(features)


bpfo_condition_df = pd.DataFrame(bpfo_condition_records)

print("Shape:", bpfo_condition_df.shape)
print("\nOperating conditions:")
print(bpfo_condition_df["operating_condition"].value_counts())

display(bpfo_condition_df.head())

In [ ]:
condition_summary = (
    bpfo_condition_df
    .groupby(["operating_condition", "fault_class"])[energy_features]
    .agg(["mean", "std"])
)

display(condition_summary.round(5))

In [ ]:
condition_mean = (
    bpfo_condition_df
    .groupby(["operating_condition", "fault_class"])[energy_features]
    .mean()
    .reset_index()
)

for feature in energy_features:

    plt.figure(figsize=(10, 5))

    for fault_class in [0, 1]:

        subset = condition_mean[
            condition_mean["fault_class"] == fault_class
        ]

        plt.plot(
            subset["operating_condition"],
            subset[feature],
            marker="o",
            label="Healthy" if fault_class == 0 else "Outer"
        )

    plt.xlabel("Operating Condition")
    plt.ylabel(feature)
    plt.title(f"{feature} Across Operating Conditions")
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
condition_speed_summary = (
    train_df
    .groupby("operating_condition")["file"]
    .first()
    .reset_index()
)

speed_rows = []

for _, row in condition_speed_summary.iterrows():

    path = find_mat_file(row["file"])

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]
    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(data, "speed")
    rpm = np.mean(speed)

    _, bpfo, bpfi, ftf, bsf = calculate_fault_frequencies(rpm)

    speed_rows.append({
        "operating_condition": row["operating_condition"],
        "rpm": rpm,
        "shaft_frequency": rpm / 60,
        "BPFO": bpfo,
        "5x_BPFO": 5 * bpfo,
        "BPFI": bpfi
    })

condition_speed_df = pd.DataFrame(speed_rows)

display(condition_speed_df.round(2))

In [ ]:
# Recompute BPFO energy features with a wider envelope-frequency range

bpfo_condition_records_500 = []

for _, row in train_df.iterrows():

    path = find_mat_file(row["file"])

    signal = get_vibration_signal(path)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]
    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(data, "speed")
    rpm = np.mean(speed)

    features = extract_bpfo_energy_features(
        signal,
        rpm,
        max_envelope_freq=500
    )

    features.update({
        "bearing": row["bearing"],
        "operating_condition": row["operating_condition"],
        "measurement_number": row["measurement_number"],
        "fault_class": row["fault_class"]
    })

    bpfo_condition_records_500.append(features)

bpfo_condition_df_500 = pd.DataFrame(bpfo_condition_records_500)

display(bpfo_condition_df_500.head())

In [ ]:
# Step 6Q
# Create harmonic-shape features using all 5 BPFO harmonics

harmonic_df = bpfo_condition_df_500.copy()

eps = 1e-12

# Harmonic ratios relative to 1x BPFO
for harmonic in range(2, 6):
    harmonic_df[f"bpfo_{harmonic}x_to_1x"] = (
        harmonic_df[f"bpfo_{harmonic}x_energy_ratio"] /
        (harmonic_df["bpfo_1x_energy_ratio"] + eps)
    )

# Combined energy across all five harmonics
harmonic_df["bpfo_1to5_total"] = (
    harmonic_df["bpfo_1x_energy_ratio"] +
    harmonic_df["bpfo_2x_energy_ratio"] +
    harmonic_df["bpfo_3x_energy_ratio"] +
    harmonic_df["bpfo_4x_energy_ratio"] +
    harmonic_df["bpfo_5x_energy_ratio"]
)

harmonic_features = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_2x_to_1x",
    "bpfo_3x_to_1x",
    "bpfo_4x_to_1x",
    "bpfo_5x_to_1x",
    "bpfo_1to5_total"
]

display(
    harmonic_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number",
            "fault_class"
        ] + harmonic_features
    ].head(10).round(5)
)

In [ ]:
harmonic_class_summary = (
    harmonic_df
    .groupby("fault_class")[harmonic_features]
    .agg(["mean", "std"])
)

display(harmonic_class_summary.round(5))

In [ ]:
harmonic_bearing_summary = (
    harmonic_df
    .groupby(["bearing", "fault_class"])[harmonic_features]
    .mean()
    .reset_index()
)

display(harmonic_bearing_summary.round(5))

In [ ]:
# Step 6R
# Stable harmonic profile: each harmonic divided by
# the total energy across all five BPFO harmonics.

stable_harmonic_df = harmonic_df.copy()

harmonic_energy_cols = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio"
]

total = stable_harmonic_df["bpfo_1to5_total"] + 1e-12

for i, col in enumerate(harmonic_energy_cols, start=1):
    stable_harmonic_df[f"bpfo_{i}x_profile"] = (
        stable_harmonic_df[col] / total
    )

stable_profile_features = [
    "bpfo_1x_profile",
    "bpfo_2x_profile",
    "bpfo_3x_profile",
    "bpfo_4x_profile",
    "bpfo_5x_profile"
]

display(
    stable_harmonic_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number",
            "fault_class"
        ] + stable_profile_features + ["bpfo_1to5_total"]
    ].head(10).round(5)
)

In [ ]:
stable_bearing_summary = (
    stable_harmonic_df
    .groupby(["bearing", "fault_class"])[
        stable_profile_features + ["bpfo_1to5_total"]
    ]
    .mean()
    .reset_index()
)

display(stable_bearing_summary.round(5))

In [ ]:
stable_class_summary = (
    stable_harmonic_df
    .groupby("fault_class")[
        stable_profile_features + ["bpfo_1to5_total"]
    ]
    .agg(["mean", "std"])
)

display(stable_class_summary.round(5))

In [ ]:
BPFO_FEATURES = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_1to5_total",
]

# Keep the normalized harmonic profile as secondary candidates
BPFO_PROFILE_FEATURES = [
    "bpfo_1x_profile",
    "bpfo_2x_profile",
    "bpfo_3x_profile",
    "bpfo_4x_profile",
    "bpfo_5x_profile",
]

print("Primary BPFO features:")
print(BPFO_FEATURES)

print("\nSecondary profile features:")
print(BPFO_PROFILE_FEATURES)

# 5. Engineer BPFO features for Healthy vs Outer

Calculate BPFO harmonic-energy features, inspect stability across bearings and operating conditions, merge them with the original 22 features, and compare candidate Healthy-vs-Outer representations.

In [ ]:
def compute_fault_aware_features(df_subset, max_envelope_freq=500):
    records = []

    for _, row in df_subset.iterrows():

        path = find_mat_file(row["file"])
        signal = get_vibration_signal(path)

        mat = sio.loadmat(
            path,
            squeeze_me=False,
            struct_as_record=False
        )

        data_keys = [key for key in mat.keys() if not key.startswith("__")]
        data = mat[data_keys[0]][0, 0]

        speed = extract_channel(data, "speed")
        rpm = np.mean(speed)

        # Primary BPFO energy features
        features = extract_bpfo_energy_features(
            signal,
            rpm,
            max_envelope_freq=max_envelope_freq
        )

        # Total BPFO energy
        total = (
            features["bpfo_1x_energy_ratio"] +
            features["bpfo_2x_energy_ratio"] +
            features["bpfo_3x_energy_ratio"] +
            features["bpfo_4x_energy_ratio"] +
            features["bpfo_5x_energy_ratio"]
        )

        features["bpfo_1to5_total"] = total

        # Stable harmonic profile
        for i in range(1, 6):
            features[f"bpfo_{i}x_profile"] = (
                features[f"bpfo_{i}x_energy_ratio"] /
                (total + 1e-12)
            )

        # Metadata
        features["file"] = row["file"]
        features["bearing"] = row["bearing"]
        features["operating_condition"] = row["operating_condition"]
        features["measurement_number"] = row["measurement_number"]
        features["fault_class"] = row["fault_class"]
        features["binary_label"] = row["binary_label"]

        records.append(features)

    return pd.DataFrame(records)

In [ ]:
print("Computing validation BPFO features...")

bpfo_val_df = compute_fault_aware_features(
    val_df,
    max_envelope_freq=500
)

print("Validation shape:", bpfo_val_df.shape)

display(bpfo_val_df.head())

In [ ]:
print("Computing test BPFO features...")

bpfo_test_df = compute_fault_aware_features(
    test_df,
    max_envelope_freq=500
)

print("Test shape:", bpfo_test_df.shape)

display(bpfo_test_df.head())

In [ ]:
# Step 6U
# Build the candidate feature sets for Healthy vs Outer classification.

PRIMARY_FAULT_FEATURES = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_1to5_total",
]

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------

train_bpfo_features = harmonic_df[
    [
        "bearing",
        "operating_condition",
        "measurement_number"
    ] + PRIMARY_FAULT_FEATURES
].copy()

train_fault_aware_df = train_df.merge(
    train_bpfo_features,
    on=["bearing", "operating_condition", "measurement_number"],
    how="inner",
    validate="one_to_one"
)

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

val_fault_aware_df = val_df.merge(
    bpfo_val_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number"
        ] + PRIMARY_FAULT_FEATURES
    ],
    on=["bearing", "operating_condition", "measurement_number"],
    how="inner",
    validate="one_to_one"
)

# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

test_fault_aware_df = test_df.merge(
    bpfo_test_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number"
        ] + PRIMARY_FAULT_FEATURES
    ],
    on=["bearing", "operating_condition", "measurement_number"],
    how="inner",
    validate="one_to_one"
)

print("Train:", train_fault_aware_df.shape)
print("Validation:", val_fault_aware_df.shape)
print("Test:", test_fault_aware_df.shape)

print("\nTrain bearing count:",
      train_fault_aware_df["bearing"].nunique())

print("Validation bearing count:",
      val_fault_aware_df["bearing"].nunique())

print("Test bearing count:",
      test_fault_aware_df["bearing"].nunique())

print("\nMissing values:")
print(
    train_fault_aware_df[
        PRIMARY_FAULT_FEATURES
    ].isna().sum()
)

In [ ]:
# Step 6U-1
# Define the two representations.

FEATURES_ORIGINAL = FEATURES_22

FEATURES_FAULT_AWARE = (
    FEATURES_22 +
    PRIMARY_FAULT_FEATURES
)

print("Original representation:")
print(len(FEATURES_ORIGINAL), "features")

print("\nFault-aware representation:")
print(len(FEATURES_FAULT_AWARE), "features")

print("\nAdded features:")
print(PRIMARY_FAULT_FEATURES)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

X_train_original = train_fault_aware_df[FEATURES_ORIGINAL]
X_val_original = val_fault_aware_df[FEATURES_ORIGINAL]

X_train_fault = train_fault_aware_df[FEATURES_FAULT_AWARE]
X_val_fault = val_fault_aware_df[FEATURES_FAULT_AWARE]

y_train_branch = train_fault_aware_df["binary_label"]
y_val_branch = val_fault_aware_df["binary_label"]


def evaluate_logistic_regression(
    X_train,
    X_val,
    y_train,
    y_val,
    feature_name
):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)
    y_prob = model.predict_proba(X_val_scaled)[:, 1]

    results = {
        "representation": feature_name,
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_prob),
        "confusion_matrix": confusion_matrix(y_val, y_pred)
    }

    return results


original_result = evaluate_logistic_regression(
    X_train_original,
    X_val_original,
    y_train_branch,
    y_val_branch,
    "Original 22"
)

fault_result = evaluate_logistic_regression(
    X_train_fault,
    X_val_fault,
    y_train_branch,
    y_val_branch,
    "Fault-aware 28"
)

print("ORIGINAL 22 FEATURES")
print("Accuracy:", original_result["accuracy"])
print("Balanced accuracy:", original_result["balanced_accuracy"])
print("F1:", original_result["f1"])
print("ROC-AUC:", original_result["roc_auc"])
print("Confusion matrix:")
print(original_result["confusion_matrix"])

print("\n" + "=" * 50 + "\n")

print("FAULT-AWARE 28 FEATURES")
print("Accuracy:", fault_result["accuracy"])
print("Balanced accuracy:", fault_result["balanced_accuracy"])
print("F1:", fault_result["f1"])
print("ROC-AUC:", fault_result["roc_auc"])
print("Confusion matrix:")
print(fault_result["confusion_matrix"])

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def train_and_predict(
    X_train,
    X_val,
    y_train
):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_val_scaled)
    probabilities = model.predict_proba(X_val_scaled)[:, 1]

    return model, scaler, predictions, probabilities


# ---------------------------------------------------------
# Original 22
# ---------------------------------------------------------

original_model, original_scaler, original_pred, original_prob = train_and_predict(
    X_train_original,
    X_val_original,
    y_train_branch
)


# ---------------------------------------------------------
# Fault-aware 28
# ---------------------------------------------------------

fault_model, fault_scaler, fault_pred, fault_prob = train_and_predict(
    X_train_fault,
    X_val_fault,
    y_train_branch
)


# Attach predictions to validation dataframe
validation_results = val_fault_aware_df[
    [
        "file",
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

validation_results["original_prediction"] = original_pred
validation_results["fault_aware_prediction"] = fault_pred


# ---------------------------------------------------------
# Per-bearing summary
# ---------------------------------------------------------

bearing_results = []

for bearing, group in validation_results.groupby("bearing"):

    true_label = group["binary_label"].iloc[0]

    original_acc = accuracy_score(
        group["binary_label"],
        group["original_prediction"]
    )

    fault_aware_acc = accuracy_score(
        group["binary_label"],
        group["fault_aware_prediction"]
    )

    original_errors = (
        group["binary_label"] != group["original_prediction"]
    ).sum()

    fault_aware_errors = (
        group["binary_label"] != group["fault_aware_prediction"]
    ).sum()

    bearing_results.append({
        "bearing": bearing,
        "class": "Healthy" if true_label == 0 else "Outer",
        "samples": len(group),
        "original_accuracy": original_acc,
        "fault_aware_accuracy": fault_aware_acc,
        "original_errors": original_errors,
        "fault_aware_errors": fault_aware_errors
    })

bearing_results_df = pd.DataFrame(bearing_results)

display(
    bearing_results_df
    .sort_values("bearing")
    .round(4)
)

In [ ]:
condition_results = (
    validation_results
    .groupby(["bearing", "operating_condition", "fault_class"])
    .apply(
        lambda g: pd.Series({
            "samples": len(g),
            "original_accuracy": accuracy_score(
                g["binary_label"],
                g["original_prediction"]
            ),
            "fault_aware_accuracy": accuracy_score(
                g["binary_label"],
                g["fault_aware_prediction"]
            ),
            "original_errors": (
                g["binary_label"] != g["original_prediction"]
            ).sum(),
            "fault_aware_errors": (
                g["binary_label"] != g["fault_aware_prediction"]
            ).sum()
        })
    )
    .reset_index()
)

display(
    condition_results
    .sort_values(["bearing", "operating_condition"])
    .round(4)
)

In [ ]:
# Step 6Y - corrected
CONDITION_TO_CHECK = "N15_M07_F04"
TARGET_BEARING = "KA08"

compare_features = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_1to5_total"
]

# ---------------------------------------------
# Training Outer bearings under same condition
# ---------------------------------------------

train_outer_condition = bpfo_condition_df_500[
    (bpfo_condition_df_500["fault_class"] == 1) &
    (bpfo_condition_df_500["operating_condition"] == CONDITION_TO_CHECK)
].copy()

# Create total BPFO energy because this dataframe
# was generated before the total column was added.
train_outer_condition["bpfo_1to5_total"] = (
    train_outer_condition["bpfo_1x_energy_ratio"] +
    train_outer_condition["bpfo_2x_energy_ratio"] +
    train_outer_condition["bpfo_3x_energy_ratio"] +
    train_outer_condition["bpfo_4x_energy_ratio"] +
    train_outer_condition["bpfo_5x_energy_ratio"]
)

# ---------------------------------------------
# Validation KA08 under same condition
# ---------------------------------------------

ka08_condition = bpfo_val_df[
    (bpfo_val_df["bearing"] == TARGET_BEARING) &
    (bpfo_val_df["operating_condition"] == CONDITION_TO_CHECK)
].copy()

print("Training Outer samples:", len(train_outer_condition))
print("KA08 samples:", len(ka08_condition))

print("\nTraining Outer summary:")
display(
    train_outer_condition[compare_features]
    .describe()
    .round(5)
)

print("\nKA08 summary:")
display(
    ka08_condition[compare_features]
    .describe()
    .round(5)
)

In [ ]:
feature = "bpfo_1to5_total"

plt.figure(figsize=(10, 5))

plt.hist(
    train_outer_condition[feature],
    bins=15,
    alpha=0.7,
    label="Training Outer"
)

plt.hist(
    ka08_condition[feature],
    bins=15,
    alpha=0.7,
    label="KA08 Outer"
)

plt.xlabel(feature)
plt.ylabel("Count")
plt.title("BPFO Total Energy: Training Outer vs KA08")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 6Z
# Evaluate BPFO-aware features alone

X_train_bpfo = train_fault_aware_df[PRIMARY_FAULT_FEATURES]
X_val_bpfo = val_fault_aware_df[PRIMARY_FAULT_FEATURES]

bpfo_only_result = evaluate_logistic_regression(
    X_train_bpfo,
    X_val_bpfo,
    y_train_branch,
    y_val_branch,
    "BPFO only"
)

print("BPFO-ONLY FEATURES")
print("Accuracy:", bpfo_only_result["accuracy"])
print("Balanced accuracy:", bpfo_only_result["balanced_accuracy"])
print("F1:", bpfo_only_result["f1"])
print("ROC-AUC:", bpfo_only_result["roc_auc"])
print("Confusion matrix:")
print(bpfo_only_result["confusion_matrix"])

In [ ]:
# Step 6Z-1
bpfo_scaler = StandardScaler()

X_train_bpfo_scaled = bpfo_scaler.fit_transform(X_train_bpfo)
X_val_bpfo_scaled = bpfo_scaler.transform(X_val_bpfo)

bpfo_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

bpfo_model.fit(
    X_train_bpfo_scaled,
    y_train_branch
)

bpfo_pred = bpfo_model.predict(X_val_bpfo_scaled)

bpfo_validation_results = val_fault_aware_df[
    [
        "bearing",
        "operating_condition",
        "fault_class",
        "binary_label"
    ]
].copy()

bpfo_validation_results["bpfo_prediction"] = bpfo_pred

bpfo_bearing_results = []

for bearing, group in bpfo_validation_results.groupby("bearing"):

    bpfo_bearing_results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Outer"
        ),
        "samples": len(group),
        "bpfo_accuracy": accuracy_score(
            group["binary_label"],
            group["bpfo_prediction"]
        ),
        "bpfo_errors": (
            group["binary_label"] != group["bpfo_prediction"]
        ).sum()
    })

display(
    pd.DataFrame(bpfo_bearing_results).round(4)
)

In [ ]:
# Step 6ZA
# Ablation study:
# Original 22
# Original + BPFO total
# Original + five BPFO harmonics
# Original + five harmonics + total

feature_sets = {
    "Original 22": FEATURES_22,

    "Original + BPFO total": (
        FEATURES_22 +
        ["bpfo_1to5_total"]
    ),

    "Original + 5 BPFO harmonics": (
        FEATURES_22 +
        [
            "bpfo_1x_energy_ratio",
            "bpfo_2x_energy_ratio",
            "bpfo_3x_energy_ratio",
            "bpfo_4x_energy_ratio",
            "bpfo_5x_energy_ratio"
        ]
    ),

    "Original + BPFO all 6": FEATURES_FAULT_AWARE
}

ablation_results = []

for name, features in feature_sets.items():

    result = evaluate_logistic_regression(
        train_fault_aware_df[features],
        val_fault_aware_df[features],
        y_train_branch,
        y_val_branch,
        name
    )

    ablation_results.append({
        "representation": name,
        "features": len(features),
        "accuracy": result["accuracy"],
        "balanced_accuracy": result["balanced_accuracy"],
        "f1": result["f1"],
        "roc_auc": result["roc_auc"]
    })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.round(4)
)

In [ ]:
# Step 6ZB
# Final validation check for the selected 23-feature representation

SELECTED_FEATURES = FEATURES_22 + ["bpfo_1to5_total"]

X_train_selected = train_fault_aware_df[SELECTED_FEATURES]
X_val_selected = val_fault_aware_df[SELECTED_FEATURES]

selected_scaler = StandardScaler()

X_train_selected_scaled = selected_scaler.fit_transform(
    X_train_selected
)

X_val_selected_scaled = selected_scaler.transform(
    X_val_selected
)

selected_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

selected_model.fit(
    X_train_selected_scaled,
    y_train_branch
)

selected_pred = selected_model.predict(
    X_val_selected_scaled
)

selected_prob = selected_model.predict_proba(
    X_val_selected_scaled
)[:, 1]

print("Selected representation: Original 22 + BPFO total")
print("Accuracy:",
      accuracy_score(y_val_branch, selected_pred))
print("Balanced accuracy:",
      balanced_accuracy_score(y_val_branch, selected_pred))
print("F1:",
      f1_score(y_val_branch, selected_pred))
print("ROC-AUC:",
      roc_auc_score(y_val_branch, selected_prob))

print("\nConfusion matrix:")
print(confusion_matrix(y_val_branch, selected_pred))

In [ ]:
selected_validation = val_fault_aware_df[
    [
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

selected_validation["prediction"] = selected_pred

selected_bearing_results = []

for bearing, group in selected_validation.groupby("bearing"):

    selected_bearing_results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Outer"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] != group["prediction"]
        ).sum()
    })

display(
    pd.DataFrame(selected_bearing_results)
    .sort_values("bearing")
    .round(4)
)

In [ ]:
# Step 6ZC
# FINAL held-out evaluation
# Selected representation:
# Original 22 + BPFO total

SELECTED_FEATURES = FEATURES_22 + ["bpfo_1to5_total"]

X_train_final = train_fault_aware_df[SELECTED_FEATURES]
X_test_final = test_fault_aware_df[SELECTED_FEATURES]

y_train_final = train_fault_aware_df["binary_label"]
y_test_final = test_fault_aware_df["binary_label"]

# Scale using TRAIN ONLY
final_scaler = StandardScaler()

X_train_final_scaled = final_scaler.fit_transform(
    X_train_final
)

X_test_final_scaled = final_scaler.transform(
    X_test_final
)

# Train final Logistic Regression
final_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

final_model.fit(
    X_train_final_scaled,
    y_train_final
)

# Test predictions
test_pred = final_model.predict(X_test_final_scaled)
test_prob = final_model.predict_proba(X_test_final_scaled)[:, 1]

# Overall metrics
print("FINAL TEST RESULTS")
print("=" * 50)

print("Accuracy:",
      accuracy_score(y_test_final, test_pred))

print("Balanced accuracy:",
      balanced_accuracy_score(y_test_final, test_pred))

print("F1:",
      f1_score(y_test_final, test_pred))

print("ROC-AUC:",
      roc_auc_score(y_test_final, test_prob))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_final, test_pred))

In [ ]:
# Final test performance by unseen bearing

test_results = test_fault_aware_df[
    [
        "file",
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

test_results["prediction"] = test_pred

final_bearing_results = []

for bearing, group in test_results.groupby("bearing"):

    final_bearing_results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Outer"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] != group["prediction"]
        ).sum()
    })

display(
    pd.DataFrame(final_bearing_results)
    .sort_values("bearing")
    .round(4)
)

# 6. Engineer BPFI features for Healthy vs Inner

Create a separate Healthy-vs-Inner dataset, calculate BPFI energy features, compare feature sets, and select thresholds using training-bearing grouped out-of-fold predictions before validation/test reporting.

In [ ]:
# Step 7A
# Healthy vs Inner binary experiment
# Both is excluded.

df_healthy_inner = df[
    df["fault_class"].isin([0, 2])
].copy()

# Binary labels:
# Healthy = 0
# Inner   = 1

df_healthy_inner["binary_label"] = (
    df_healthy_inner["fault_class"] == 2
).astype(int)

TRAIN_BEARINGS_INNER = [
    "K001", "K002", "K003", "K004",
    "KI01", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17"
]

VAL_BEARINGS_INNER = [
    "K005",
    "KI03"
]

TEST_BEARINGS_INNER = [
    "K006",
    "KI18", "KI21"
]


def assign_inner_split(bearing):
    if bearing in TRAIN_BEARINGS_INNER:
        return "train"
    elif bearing in VAL_BEARINGS_INNER:
        return "val"
    elif bearing in TEST_BEARINGS_INNER:
        return "test"
    else:
        return None


df_healthy_inner["split"] = (
    df_healthy_inner["bearing"]
    .map(assign_inner_split)
)

print("Dataset shape:", df_healthy_inner.shape)

print("\nFault distribution:")
print(df_healthy_inner["fault_class"].value_counts())

print("\nSplit distribution:")
print(df_healthy_inner["split"].value_counts())

print("\nBearing distribution:")
print(
    df_healthy_inner
    .groupby(["split", "fault_class"])["bearing"]
    .nunique()
)

print("\nUnassigned rows:")
print(df_healthy_inner["split"].isna().sum())

In [ ]:
# Step 7B
# Create train / validation / test dataframes

train_inner_df = df_healthy_inner[
    df_healthy_inner["split"] == "train"
].copy()

val_inner_df = df_healthy_inner[
    df_healthy_inner["split"] == "val"
].copy()

test_inner_df = df_healthy_inner[
    df_healthy_inner["split"] == "test"
].copy()


print("Train shape:", train_inner_df.shape)
print("Validation shape:", val_inner_df.shape)
print("Test shape:", test_inner_df.shape)

print("\nTrain class distribution:")
print(train_inner_df["binary_label"].value_counts())

print("\nValidation class distribution:")
print(val_inner_df["binary_label"].value_counts())

print("\nTest class distribution:")
print(test_inner_df["binary_label"].value_counts())

print("\nTrain bearings:")
print(sorted(train_inner_df["bearing"].unique()))

print("\nValidation bearings:")
print(sorted(val_inner_df["bearing"].unique()))

print("\nTest bearings:")
print(sorted(test_inner_df["bearing"].unique()))

In [ ]:
# Step 7C
# Original 22-feature representation for Healthy vs Inner

X_train_inner_original = train_inner_df[FEATURES_22]
y_train_inner = train_inner_df["binary_label"]

X_val_inner_original = val_inner_df[FEATURES_22]
y_val_inner = val_inner_df["binary_label"]

X_test_inner_original = test_inner_df[FEATURES_22]
y_test_inner = test_inner_df["binary_label"]

print("Train X:", X_train_inner_original.shape)
print("Validation X:", X_val_inner_original.shape)
print("Test X:", X_test_inner_original.shape)

In [ ]:
representative_inner_bearings = [
    "K001",   # Healthy
    "KI01",   # Inner
    "KI04",   # Inner
    "KI05"    # Inner
]

for bearing in representative_inner_bearings:

    rows = train_inner_df[
        train_inner_df["bearing"] == bearing
    ]

    print(f"\n{bearing}")

    print(
        rows[
            [
                "file",
                "operating_condition",
                "measurement_number",
                "fault_class"
            ]
        ]
        .head(8)
        .to_string(index=False)
    )

In [ ]:
representative_inner_files = {
    "K001": "N09_M07_F10_K001_1.mat",   # Healthy
    "KI01": "N09_M07_F10_KI01_1.mat",   # Inner
    "KI04": "N09_M07_F10_KI04_1.mat",   # Inner
    "KI05": "N09_M07_F10_KI05_1.mat"    # Inner
}

inner_raw_signals = {}

for bearing, filename in representative_inner_files.items():

    path = find_mat_file(filename)
    signal = get_vibration_signal(path)

    inner_raw_signals[bearing] = signal

    print(f"\n{bearing}")
    print("Samples:", len(signal))
    print("Min:", signal.min())
    print("Max:", signal.max())
    print("Mean:", signal.mean())
    print("Std:", signal.std())
    print("RMS:", np.sqrt(np.mean(signal ** 2)))

In [ ]:
WINDOW_SECONDS = 0.01
N_SAMPLES = int(WINDOW_SECONDS * FS)

for bearing, signal in inner_raw_signals.items():

    segment = signal[:N_SAMPLES]
    time = np.arange(len(segment)) / FS

    plt.figure(figsize=(12, 4))
    plt.plot(time, segment)

    plt.title(f"{bearing} - Raw Vibration")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
INNER_REFERENCE_FILES = {
    "K001": "N09_M07_F10_K001_1.mat",
    "KI01": "N09_M07_F10_KI01_1.mat",
    "KI04": "N09_M07_F10_KI04_1.mat",
    "KI05": "N09_M07_F10_KI05_1.mat"
}

inner_frequency_info = []

for bearing, filename in INNER_REFERENCE_FILES.items():

    path = find_mat_file(filename)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [key for key in mat.keys() if not key.startswith("__")]
    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(data, "speed")
    rpm = np.mean(speed)

    fr, bpfo, bpfi, ftf, bsf = calculate_fault_frequencies(rpm)

    inner_frequency_info.append({
        "bearing": bearing,
        "rpm": rpm,
        "shaft_frequency": fr,
        "BPFO": bpfo,
        "BPFI": bpfi,
        "FTF": ftf,
        "BSF": bsf
    })

inner_frequency_df = pd.DataFrame(inner_frequency_info)

display(inner_frequency_df.round(3))

In [ ]:
plt.figure(figsize=(14, 6))

# Use the measured BPFI from the approximately 900 rpm condition
reference_bearing = "KI01"

reference_bpfi = inner_frequency_df.loc[
    inner_frequency_df["bearing"] == reference_bearing,
    "BPFI"
].iloc[0]

for bearing, signal in inner_raw_signals.items():

    frequencies, spectrum = envelope_spectrum(
        signal,
        FS,
        LOWCUT,
        HIGHCUT,
        500
    )

    spectrum = spectrum / (np.max(spectrum) + 1e-12)

    plt.plot(
        frequencies,
        spectrum,
        label=bearing
    )

# BPFI harmonics
for harmonic in range(1, 6):

    freq = harmonic * reference_bpfi

    plt.axvline(
        freq,
        linestyle="--",
        alpha=0.5
    )

    plt.text(
        freq,
        1.02,
        f"{harmonic}×BPFI",
        rotation=90,
        ha="center",
        va="bottom"
    )

plt.xlim(0, 500)
plt.ylim(0, 1.1)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized Envelope Magnitude")
plt.title("Inner-Fault Envelope Spectrum with Expected BPFI Harmonics")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def extract_bpfi_energy_features(
    signal,
    rpm,
    fs=64000,
    lowcut=2000,
    highcut=10000,
    max_envelope_freq=500,
    tolerance=3
):
    frequencies, spectrum = envelope_spectrum(
        signal,
        fs,
        lowcut,
        highcut,
        max_envelope_freq
    )

    spectral_power = spectrum ** 2

    total_band = (
        (frequencies >= 20) &
        (frequencies <= max_envelope_freq)
    )

    total_energy = np.sum(
        spectral_power[total_band]
    )

    _, _, bpfi, _, _ = calculate_fault_frequencies(rpm)

    features = {}

    for harmonic in range(1, 6):

        target = harmonic * bpfi

        band = np.abs(
            frequencies - target
        ) <= tolerance

        harmonic_energy = np.sum(
            spectral_power[band]
        )

        features[
            f"bpfi_{harmonic}x_energy_ratio"
        ] = (
            harmonic_energy /
            (total_energy + 1e-12)
        )

    features["bpfi_1to5_total"] = sum(
        features[
            f"bpfi_{harmonic}x_energy_ratio"
        ]
        for harmonic in range(1, 6)
    )

    return features

In [ ]:
# Compute BPFI features across all training Healthy + Inner recordings

bpfi_train_records = []

for _, row in train_inner_df.iterrows():

    path = find_mat_file(row["file"])

    signal = get_vibration_signal(path)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(
        data,
        "speed"
    )

    rpm = np.mean(speed)

    features = extract_bpfi_energy_features(
        signal,
        rpm
    )

    features.update({
        "bearing": row["bearing"],
        "operating_condition": row["operating_condition"],
        "measurement_number": row["measurement_number"],
        "fault_class": row["fault_class"]
    })

    bpfi_train_records.append(features)


bpfi_train_df = pd.DataFrame(
    bpfi_train_records
)

print("Shape:", bpfi_train_df.shape)

display(
    bpfi_train_df.head()
)

In [ ]:
BPFI_FEATURES = [
    "bpfi_1x_energy_ratio",
    "bpfi_2x_energy_ratio",
    "bpfi_3x_energy_ratio",
    "bpfi_4x_energy_ratio",
    "bpfi_5x_energy_ratio",
    "bpfi_1to5_total"
]

bpfi_class_summary = (
    bpfi_train_df
    .groupby("fault_class")[BPFI_FEATURES]
    .agg(["mean", "std"])
)

display(
    bpfi_class_summary.round(5)
)

In [ ]:
bpfi_bearing_summary = (
    bpfi_train_df
    .groupby(["bearing", "fault_class"])[BPFI_FEATURES]
    .agg(["mean", "std"])
)

display(
    bpfi_bearing_summary.round(5)
)

In [ ]:
bpfi_condition_summary = (
    bpfi_train_df
    .groupby(["operating_condition", "fault_class"])[BPFI_FEATURES]
    .agg(["mean", "std"])
)

display(
    bpfi_condition_summary.round(5)
)

In [ ]:
bpfi_condition_mean = (
    bpfi_train_df
    .groupby(["operating_condition", "fault_class"])[BPFI_FEATURES]
    .mean()
    .reset_index()
)

for feature in BPFI_FEATURES:

    plt.figure(figsize=(10, 5))

    for fault_class in [0, 2]:

        subset = bpfi_condition_mean[
            bpfi_condition_mean["fault_class"] == fault_class
        ]

        plt.plot(
            subset["operating_condition"],
            subset[feature],
            marker="o",
            label="Healthy" if fault_class == 0 else "Inner"
        )

    plt.xlabel("Operating Condition")
    plt.ylabel(feature)
    plt.title(f"{feature} Across Operating Conditions")
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 7K-1
# Recalculate BPFI features with enough frequency range
# to include 5x BPFI at the ~1500 rpm operating conditions.

bpfi_train_records_700 = []

for _, row in train_inner_df.iterrows():

    path = find_mat_file(row["file"])

    signal = get_vibration_signal(path)

    mat = sio.loadmat(
        path,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    data = mat[data_keys[0]][0, 0]

    speed = extract_channel(
        data,
        "speed"
    )

    rpm = np.mean(speed)

    features = extract_bpfi_energy_features(
        signal,
        rpm,
        max_envelope_freq=700
    )

    features.update({
        "bearing": row["bearing"],
        "operating_condition": row["operating_condition"],
        "measurement_number": row["measurement_number"],
        "fault_class": row["fault_class"]
    })

    bpfi_train_records_700.append(features)


bpfi_train_df_700 = pd.DataFrame(
    bpfi_train_records_700
)

print("Shape:", bpfi_train_df_700.shape)

display(
    bpfi_train_df_700.head()
)

In [ ]:
bpfi_condition_summary_700 = (
    bpfi_train_df_700
    .groupby(["operating_condition", "fault_class"])[BPFI_FEATURES]
    .agg(["mean", "std"])
)

display(
    bpfi_condition_summary_700.round(5)
)

In [ ]:
# Step 7L
# Bearing-level BPFI summary using the corrected 700 Hz analysis

bpfi_bearing_summary_700 = (
    bpfi_train_df_700
    .groupby(["bearing", "fault_class"])[BPFI_FEATURES]
    .agg(["mean", "std"])
)

display(
    bpfi_bearing_summary_700.round(5)
)

In [ ]:
# Focus specifically on total BPFI-related energy

bpfi_total_bearing = (
    bpfi_train_df_700
    .groupby(["bearing", "fault_class"])["bpfi_1to5_total"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

display(
    bpfi_total_bearing.round(5)
)

In [ ]:
# Step 7M
# Compute corrected BPFI features for validation and test.
# Uses 700 Hz envelope range so all 1x-5x BPFI components
# are included at the ~1500 rpm operating conditions.

def compute_bpfi_features(df_subset, max_envelope_freq=700):

    records = []

    for _, row in df_subset.iterrows():

        path = find_mat_file(row["file"])

        signal = get_vibration_signal(path)

        mat = sio.loadmat(
            path,
            squeeze_me=False,
            struct_as_record=False
        )

        data_keys = [
            key for key in mat.keys()
            if not key.startswith("__")
        ]

        data = mat[data_keys[0]][0, 0]

        speed = extract_channel(
            data,
            "speed"
        )

        rpm = np.mean(speed)

        features = extract_bpfi_energy_features(
            signal,
            rpm,
            max_envelope_freq=max_envelope_freq
        )

        features.update({
            "file": row["file"],
            "bearing": row["bearing"],
            "operating_condition": row["operating_condition"],
            "measurement_number": row["measurement_number"],
            "fault_class": row["fault_class"],
            "binary_label": row["binary_label"]
        })

        records.append(features)

    return pd.DataFrame(records)


print("Computing validation BPFI features...")

bpfi_val_df = compute_bpfi_features(
    val_inner_df,
    max_envelope_freq=700
)

print("Validation shape:", bpfi_val_df.shape)


print("\nComputing test BPFI features...")

bpfi_test_df = compute_bpfi_features(
    test_inner_df,
    max_envelope_freq=700
)

print("Test shape:", bpfi_test_df.shape)

print("\nValidation bearings:")
print(sorted(bpfi_val_df["bearing"].unique()))

print("\nTest bearings:")
print(sorted(bpfi_test_df["bearing"].unique()))

In [ ]:
# Add the total BPFI feature explicitly to the corrected training dataframe

BPFI_PRIMARY_FEATURES = [
    "bpfi_1x_energy_ratio",
    "bpfi_2x_energy_ratio",
    "bpfi_3x_energy_ratio",
    "bpfi_4x_energy_ratio",
    "bpfi_5x_energy_ratio",
    "bpfi_1to5_total"
]

print("BPFI primary features:")
print(BPFI_PRIMARY_FEATURES)

print("\nTraining:", bpfi_train_df_700.shape)
print("Validation:", bpfi_val_df.shape)
print("Test:", bpfi_test_df.shape)

print("\nMissing values in training:")
print(
    bpfi_train_df_700[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

print("\nMissing values in validation:")
print(
    bpfi_val_df[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

print("\nMissing values in test:")
print(
    bpfi_test_df[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

In [ ]:
# Step 7N
# Merge corrected BPFI features into the Healthy-vs-Inner datasets

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------

train_bpfi_features = bpfi_train_df_700[
    [
        "bearing",
        "operating_condition",
        "measurement_number"
    ] + BPFI_PRIMARY_FEATURES
].copy()

train_inner_fault_aware_df = train_inner_df.merge(
    train_bpfi_features,
    on=[
        "bearing",
        "operating_condition",
        "measurement_number"
    ],
    how="inner",
    validate="one_to_one"
)

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

val_inner_fault_aware_df = val_inner_df.merge(
    bpfi_val_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number"
        ] + BPFI_PRIMARY_FEATURES
    ],
    on=[
        "bearing",
        "operating_condition",
        "measurement_number"
    ],
    how="inner",
    validate="one_to_one"
)

# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

test_inner_fault_aware_df = test_inner_df.merge(
    bpfi_test_df[
        [
            "bearing",
            "operating_condition",
            "measurement_number"
        ] + BPFI_PRIMARY_FEATURES
    ],
    on=[
        "bearing",
        "operating_condition",
        "measurement_number"
    ],
    how="inner",
    validate="one_to_one"
)

print("Train:", train_inner_fault_aware_df.shape)
print("Validation:", val_inner_fault_aware_df.shape)
print("Test:", test_inner_fault_aware_df.shape)

print("\nMissing BPFI values:")

print(
    train_inner_fault_aware_df[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

print(
    val_inner_fault_aware_df[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

print(
    test_inner_fault_aware_df[BPFI_PRIMARY_FEATURES]
    .isna()
    .sum()
)

In [ ]:
# Step 7N-1
# Candidate representations for Healthy vs Inner

INNER_FEATURE_SETS = {
    "Original 22": FEATURES_22,

    "Original + BPFI total": (
        FEATURES_22 +
        ["bpfi_1to5_total"]
    ),

    "Original + 5 BPFI harmonics": (
        FEATURES_22 +
        [
            "bpfi_1x_energy_ratio",
            "bpfi_2x_energy_ratio",
            "bpfi_3x_energy_ratio",
            "bpfi_4x_energy_ratio",
            "bpfi_5x_energy_ratio"
        ]
    ),

    "Original + BPFI all 6": (
        FEATURES_22 +
        BPFI_PRIMARY_FEATURES
    )
}

for name, features in INNER_FEATURE_SETS.items():
    print(f"{name}: {len(features)} features")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_train_inner_branch = train_inner_fault_aware_df["binary_label"]
y_val_inner_branch = val_inner_fault_aware_df["binary_label"]


def evaluate_inner_representation(
    train_df_input,
    val_df_input,
    features,
    feature_name
):
    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_df_input[features]
    )

    X_val = scaler.transform(
        val_df_input[features]
    )

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(
        X_train,
        y_train_inner_branch
    )

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    return {
        "representation": feature_name,
        "features": len(features),
        "accuracy": accuracy_score(
            y_val_inner_branch,
            y_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_val_inner_branch,
            y_pred
        ),
        "f1": f1_score(
            y_val_inner_branch,
            y_pred
        ),
        "roc_auc": roc_auc_score(
            y_val_inner_branch,
            y_prob
        ),
        "confusion_matrix": confusion_matrix(
            y_val_inner_branch,
            y_pred
        )
    }


inner_ablation_results = []

for name, features in INNER_FEATURE_SETS.items():

    result = evaluate_inner_representation(
        train_inner_fault_aware_df,
        val_inner_fault_aware_df,
        features,
        name
    )

    inner_ablation_results.append({
        "representation": result["representation"],
        "features": result["features"],
        "accuracy": result["accuracy"],
        "balanced_accuracy": result["balanced_accuracy"],
        "f1": result["f1"],
        "roc_auc": result["roc_auc"],
        "confusion_matrix": result["confusion_matrix"]
    })

inner_ablation_df = pd.DataFrame(
    inner_ablation_results
)

display(
    inner_ablation_df[
        [
            "representation",
            "features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "confusion_matrix"
        ]
    ].round(4)
)

In [ ]:
# Step 7P
# FINAL held-out Healthy vs Inner evaluation
# Selected representation: Original 22 features

INNER_SELECTED_FEATURES = FEATURES_22

X_train_inner_final = train_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

X_test_inner_final = test_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_train_inner_final = train_inner_fault_aware_df[
    "binary_label"
]

y_test_inner_final = test_inner_fault_aware_df[
    "binary_label"
]

# Scale using TRAIN ONLY
inner_final_scaler = StandardScaler()

X_train_inner_final_scaled = (
    inner_final_scaler.fit_transform(
        X_train_inner_final
    )
)

X_test_inner_final_scaled = (
    inner_final_scaler.transform(
        X_test_inner_final
    )
)

# Final Logistic Regression
inner_final_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

inner_final_model.fit(
    X_train_inner_final_scaled,
    y_train_inner_final
)

# Test predictions
inner_test_pred = inner_final_model.predict(
    X_test_inner_final_scaled
)

inner_test_prob = (
    inner_final_model
    .predict_proba(X_test_inner_final_scaled)[:, 1]
)

print("FINAL HEALTHY vs INNER TEST RESULTS")
print("=" * 50)

print(
    "Accuracy:",
    accuracy_score(
        y_test_inner_final,
        inner_test_pred
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_test_inner_final,
        inner_test_pred
    )
)

print(
    "F1:",
    f1_score(
        y_test_inner_final,
        inner_test_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test_inner_final,
        inner_test_prob
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test_inner_final,
        inner_test_pred
    )
)

In [ ]:
# Final test performance for each unseen bearing

inner_test_results = test_inner_fault_aware_df[
    [
        "file",
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

inner_test_results["prediction"] = inner_test_pred

inner_bearing_results = []

for bearing, group in inner_test_results.groupby(
    "bearing"
):

    inner_bearing_results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Inner"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] !=
            group["prediction"]
        ).sum()
    })

display(
    pd.DataFrame(inner_bearing_results)
    .sort_values("bearing")
    .round(4)
)

In [ ]:
# Step 7P-1
# Select the classification threshold using VALIDATION data only.
# The test set is not used here.

from sklearn.metrics import balanced_accuracy_score

# Validation data using the already-selected original 22 features
X_val_inner_final = val_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_val_inner_final = val_inner_fault_aware_df[
    "binary_label"
]

# Use the scaler/model fitted only on the training data
X_val_inner_final_scaled = (
    inner_final_scaler.transform(
        X_val_inner_final
    )
)

val_inner_prob = (
    inner_final_model
    .predict_proba(
        X_val_inner_final_scaled
    )[:, 1]
)

threshold_results = []

for threshold in np.arange(0.05, 0.951, 0.01):

    val_pred_threshold = (
        val_inner_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(
            y_val_inner_final,
            val_pred_threshold
        ),
        "accuracy": accuracy_score(
            y_val_inner_final,
            val_pred_threshold
        ),
        "f1": f1_score(
            y_val_inner_final,
            val_pred_threshold
        )
    })

threshold_df = pd.DataFrame(
    threshold_results
)

best_threshold_row = (
    threshold_df
    .sort_values(
        ["balanced_accuracy", "f1", "threshold"],
        ascending=[False, False, True]
    )
    .iloc[0]
)

print("Best validation threshold:")
print(best_threshold_row)

In [ ]:
# Step 7P-2
# Inspect the validation probability separation

validation_probability_summary = pd.DataFrame({
    "true_label": y_val_inner_final.values,
    "probability_inner": val_inner_prob
})

print("Healthy probability range:")
print(
    validation_probability_summary[
        validation_probability_summary["true_label"] == 0
    ]["probability_inner"].describe()
)

print("\nInner probability range:")
print(
    validation_probability_summary[
        validation_probability_summary["true_label"] == 1
    ]["probability_inner"].describe()
)

print("\nLowest Inner probability:")
print(
    validation_probability_summary[
        validation_probability_summary["true_label"] == 1
    ]["probability_inner"].min()
)

print("\nHighest Healthy probability:")
print(
    validation_probability_summary[
        validation_probability_summary["true_label"] == 0
    ]["probability_inner"].max()
)

In [ ]:
# Step 7P-3
# Apply the threshold selected ONLY from validation data.

SELECTED_INNER_THRESHOLD = 0.46

inner_test_pred_threshold = (
    inner_test_prob >= SELECTED_INNER_THRESHOLD
).astype(int)

print("FINAL HEALTHY vs INNER TEST RESULTS")
print("=" * 50)

print("Threshold:", SELECTED_INNER_THRESHOLD)

print(
    "Accuracy:",
    accuracy_score(
        y_test_inner_final,
        inner_test_pred_threshold
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_test_inner_final,
        inner_test_pred_threshold
    )
)

print(
    "F1:",
    f1_score(
        y_test_inner_final,
        inner_test_pred_threshold
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test_inner_final,
        inner_test_prob
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test_inner_final,
        inner_test_pred_threshold
    )
)

In [ ]:
# Per-bearing test performance using the frozen 0.46 threshold

inner_test_results_threshold = test_inner_fault_aware_df[
    [
        "file",
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

inner_test_results_threshold["prediction"] = (
    inner_test_pred_threshold
)

inner_bearing_results_threshold = []

for bearing, group in (
    inner_test_results_threshold.groupby("bearing")
):

    inner_bearing_results_threshold.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Inner"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] !=
            group["prediction"]
        ).sum()
    })

display(
    pd.DataFrame(
        inner_bearing_results_threshold
    )
    .sort_values("bearing")
    .round(4)
)

In [ ]:
from sklearn.model_selection import GroupKFold

# Step 7Q
# Group-aware out-of-fold threshold selection
# using TRAINING BEARINGS ONLY.

X_train_cv = train_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_train_cv = train_inner_fault_aware_df[
    "binary_label"
]

groups_cv = train_inner_fault_aware_df[
    "bearing"
]

gkf = GroupKFold(n_splits=4)

oof_prob = np.zeros(len(train_inner_fault_aware_df))

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        X_train_cv,
        y_train_cv,
        groups=groups_cv
    ),
    start=1
):

    X_fold_train = X_train_cv.iloc[train_idx]
    X_fold_val = X_train_cv.iloc[val_idx]

    y_fold_train = y_train_cv.iloc[train_idx]

    scaler = StandardScaler()

    X_fold_train_scaled = scaler.fit_transform(
        X_fold_train
    )

    X_fold_val_scaled = scaler.transform(
        X_fold_val
    )

    fold_model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    fold_model.fit(
        X_fold_train_scaled,
        y_fold_train
    )

    oof_prob[val_idx] = fold_model.predict_proba(
        X_fold_val_scaled
    )[:, 1]

print("Generated OOF predictions for:",
      len(oof_prob), "training samples")

print("Unique training bearings:",
      groups_cv.nunique())

In [ ]:
# Select threshold from bearing-level OOF predictions

oof_threshold_results = []

for threshold in np.arange(0.05, 0.951, 0.01):

    oof_pred = (
        oof_prob >= threshold
    ).astype(int)

    oof_threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(
            y_train_cv,
            oof_pred
        ),
        "accuracy": accuracy_score(
            y_train_cv,
            oof_pred
        ),
        "f1": f1_score(
            y_train_cv,
            oof_pred
        )
    })

oof_threshold_df = pd.DataFrame(
    oof_threshold_results
)

best_oof_threshold = (
    oof_threshold_df
    .sort_values(
        ["balanced_accuracy", "f1", "threshold"],
        ascending=[False, False, True]
    )
    .iloc[0]
)

print("Best group-CV threshold:")
print(best_oof_threshold)

In [ ]:
from sklearn.metrics import roc_auc_score

print("Group-CV ROC-AUC:",
      roc_auc_score(y_train_cv, oof_prob))

print("\nPerformance at threshold 0.50:")

oof_pred_05 = (oof_prob >= 0.50).astype(int)

print("Accuracy:",
      accuracy_score(y_train_cv, oof_pred_05))

print("Balanced accuracy:",
      balanced_accuracy_score(y_train_cv, oof_pred_05))

print("F1:",
      f1_score(y_train_cv, oof_pred_05))

print("\nConfusion matrix:")
print(confusion_matrix(y_train_cv, oof_pred_05))


print("\nPerformance at threshold 0.06:")

oof_pred_006 = (oof_prob >= 0.06).astype(int)

print("Accuracy:",
      accuracy_score(y_train_cv, oof_pred_006))

print("Balanced accuracy:",
      balanced_accuracy_score(y_train_cv, oof_pred_006))

print("F1:",
      f1_score(y_train_cv, oof_pred_006))

print("\nConfusion matrix:")
print(confusion_matrix(y_train_cv, oof_pred_006))

In [ ]:
oof_diagnostic_df = train_inner_fault_aware_df[
    [
        "bearing",
        "fault_class",
        "binary_label"
    ]
].copy()

oof_diagnostic_df["oof_probability_inner"] = oof_prob

oof_bearing_summary = (
    oof_diagnostic_df
    .groupby(["bearing", "fault_class"])["oof_probability_inner"]
    .agg(["min", "max", "mean", "std"])
    .reset_index()
)

display(
    oof_bearing_summary.round(4)
)

In [ ]:
# Step 7Q-2
# Compare Inner feature representations using
# bearing-grouped cross-validation on TRAINING bearings only.

inner_groupcv_results = []

for representation_name, feature_list in INNER_FEATURE_SETS.items():

    X = train_inner_fault_aware_df[feature_list]
    y = train_inner_fault_aware_df["binary_label"]
    groups = train_inner_fault_aware_df["bearing"]

    gkf = GroupKFold(n_splits=4)

    oof_prob_rep = np.zeros(len(train_inner_fault_aware_df))

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(X, y, groups=groups),
        start=1
    ):

        X_fold_train = X.iloc[train_idx]
        X_fold_val = X.iloc[val_idx]

        y_fold_train = y.iloc[train_idx]

        scaler = StandardScaler()

        X_fold_train_scaled = scaler.fit_transform(
            X_fold_train
        )

        X_fold_val_scaled = scaler.transform(
            X_fold_val
        )

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(
            X_fold_train_scaled,
            y_fold_train
        )

        oof_prob_rep[val_idx] = model.predict_proba(
            X_fold_val_scaled
        )[:, 1]

    # Evaluate ranking first
    oof_auc = roc_auc_score(
        y,
        oof_prob_rep
    )

    # Evaluate at the conventional threshold
    oof_pred = (
        oof_prob_rep >= 0.5
    ).astype(int)

    oof_balanced_accuracy = (
        balanced_accuracy_score(
            y,
            oof_pred
        )
    )

    oof_f1 = f1_score(
        y,
        oof_pred
    )

    inner_groupcv_results.append({
        "representation": representation_name,
        "features": len(feature_list),
        "group_cv_roc_auc": oof_auc,
        "group_cv_balanced_accuracy": oof_balanced_accuracy,
        "group_cv_f1": oof_f1
    })


inner_groupcv_df = pd.DataFrame(
    inner_groupcv_results
)

display(
    inner_groupcv_df
    .sort_values(
        "group_cv_roc_auc",
        ascending=False
    )
    .round(4)
)

In [ ]:
# Step 7R
# Generate bearing-grouped OOF probabilities for the selected
# 28-feature Inner representation.

INNER_SELECTED_FEATURES = INNER_FEATURE_SETS[
    "Original + BPFI all 6"
]

X_inner_oof = train_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_inner_oof = train_inner_fault_aware_df[
    "binary_label"
]

groups_inner_oof = train_inner_fault_aware_df[
    "bearing"
]

gkf = GroupKFold(n_splits=4)

inner_oof_prob = np.zeros(
    len(train_inner_fault_aware_df)
)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        X_inner_oof,
        y_inner_oof,
        groups=groups_inner_oof
    ),
    start=1
):

    X_fold_train = X_inner_oof.iloc[train_idx]
    X_fold_val = X_inner_oof.iloc[val_idx]

    y_fold_train = y_inner_oof.iloc[train_idx]

    scaler = StandardScaler()

    X_fold_train_scaled = scaler.fit_transform(
        X_fold_train
    )

    X_fold_val_scaled = scaler.transform(
        X_fold_val
    )

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(
        X_fold_train_scaled,
        y_fold_train
    )

    inner_oof_prob[val_idx] = (
        model.predict_proba(
            X_fold_val_scaled
        )[:, 1]
    )

print(
    "Group-CV ROC-AUC:",
    roc_auc_score(
        y_inner_oof,
        inner_oof_prob
    )
)

In [ ]:
# Select threshold using ONLY training-bearing OOF predictions.

inner_threshold_results = []

for threshold in np.arange(
    0.05,
    0.951,
    0.01
):

    pred = (
        inner_oof_prob >= threshold
    ).astype(int)

    inner_threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_inner_oof,
                pred
            )
        ),
        "accuracy": (
            accuracy_score(
                y_inner_oof,
                pred
            )
        ),
        "f1": (
            f1_score(
                y_inner_oof,
                pred
            )
        )
    })

inner_threshold_df = pd.DataFrame(
    inner_threshold_results
)

best_inner_oof_threshold = (
    inner_threshold_df
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "threshold"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .iloc[0]
)

print(
    "Best Group-CV threshold:"
)

print(
    best_inner_oof_threshold
)

In [ ]:
inner_oof_diagnostic = train_inner_fault_aware_df[
    [
        "bearing",
        "fault_class",
        "binary_label"
    ]
].copy()

inner_oof_diagnostic[
    "oof_probability_inner"
] = inner_oof_prob

selected_threshold = (
    best_inner_oof_threshold["threshold"]
)

inner_oof_diagnostic[
    "oof_prediction"
] = (
    inner_oof_diagnostic[
        "oof_probability_inner"
    ] >= selected_threshold
).astype(int)

inner_oof_bearing_summary = (
    inner_oof_diagnostic
    .groupby(
        ["bearing", "fault_class"]
    )
    .agg(
        probability_min=(
            "oof_probability_inner",
            "min"
        ),
        probability_max=(
            "oof_probability_inner",
            "max"
        ),
        probability_mean=(
            "oof_probability_inner",
            "mean"
        ),
        accuracy=(
            "oof_prediction",
            lambda x: np.mean(
                x ==
                inner_oof_diagnostic.loc[
                    x.index,
                    "binary_label"
                ]
            )
        )
    )
    .reset_index()
)

display(
    inner_oof_bearing_summary.round(4)
)

In [ ]:
# Step 7S
# External validation using the frozen:
#   28-feature representation
#   threshold = 0.87
#
# No tuning is performed here.

INNER_SELECTED_FEATURES = INNER_FEATURE_SETS[
    "Original + BPFI all 6"
]

INNER_SELECTED_THRESHOLD = 0.87

X_train_selected = train_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

X_val_selected = val_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_train_selected = train_inner_fault_aware_df[
    "binary_label"
]

y_val_selected = val_inner_fault_aware_df[
    "binary_label"
]

# Fit scaler on training bearings only
inner_selected_scaler = StandardScaler()

X_train_selected_scaled = (
    inner_selected_scaler.fit_transform(
        X_train_selected
    )
)

X_val_selected_scaled = (
    inner_selected_scaler.transform(
        X_val_selected
    )
)

# Train final model on ALL training bearings
inner_selected_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

inner_selected_model.fit(
    X_train_selected_scaled,
    y_train_selected
)

# Validation probabilities
val_selected_prob = (
    inner_selected_model
    .predict_proba(
        X_val_selected_scaled
    )[:, 1]
)

val_selected_pred = (
    val_selected_prob >= INNER_SELECTED_THRESHOLD
).astype(int)

print("FROZEN INNER VALIDATION")
print("=" * 50)

print("Features:", len(INNER_SELECTED_FEATURES))
print("Threshold:", INNER_SELECTED_THRESHOLD)

print(
    "Accuracy:",
    accuracy_score(
        y_val_selected,
        val_selected_pred
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_val_selected,
        val_selected_pred
    )
)

print(
    "F1:",
    f1_score(
        y_val_selected,
        val_selected_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_selected,
        val_selected_prob
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_val_selected,
        val_selected_pred
    )
)

In [ ]:
# Per-bearing validation performance

inner_external_validation = val_inner_fault_aware_df[
    [
        "bearing",
        "fault_class",
        "binary_label"
    ]
].copy()

inner_external_validation["prediction"] = (
    val_selected_pred
)

inner_external_validation["probability"] = (
    val_selected_prob
)

results = []

for bearing, group in (
    inner_external_validation.groupby("bearing")
):

    results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Inner"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] !=
            group["prediction"]
        ).sum(),
        "mean_probability": group[
            "probability"
        ].mean()
    })

display(
    pd.DataFrame(results)
    .sort_values("bearing")
    .round(4)
)

In [ ]:
# Step 7T
# FINAL HELD-OUT HEALTHY vs INNER TEST
#
# Frozen configuration:
#   28 features
#   threshold = 0.87
#
# Test bearings:
#   K006  -> Healthy
#   KI18  -> Inner
#   KI21  -> Inner

X_train_inner_final = train_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

X_test_inner_final = test_inner_fault_aware_df[
    INNER_SELECTED_FEATURES
]

y_train_inner_final = train_inner_fault_aware_df[
    "binary_label"
]

y_test_inner_final = test_inner_fault_aware_df[
    "binary_label"
]

# ---------------------------------------------------------
# Fit scaler on ALL training bearings only
# ---------------------------------------------------------

inner_final_scaler_28 = StandardScaler()

X_train_inner_final_scaled = (
    inner_final_scaler_28.fit_transform(
        X_train_inner_final
    )
)

X_test_inner_final_scaled = (
    inner_final_scaler_28.transform(
        X_test_inner_final
    )
)

# ---------------------------------------------------------
# Train final model
# ---------------------------------------------------------

inner_final_model_28 = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

inner_final_model_28.fit(
    X_train_inner_final_scaled,
    y_train_inner_final
)

# ---------------------------------------------------------
# Test probabilities
# ---------------------------------------------------------

inner_test_prob_28 = (
    inner_final_model_28
    .predict_proba(
        X_test_inner_final_scaled
    )[:, 1]
)

# Frozen threshold from Group-CV
INNER_FINAL_THRESHOLD = 0.87

inner_test_pred_28 = (
    inner_test_prob_28 >= INNER_FINAL_THRESHOLD
).astype(int)

# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

print("FINAL HELD-OUT HEALTHY vs INNER")
print("=" * 55)

print("Features:", len(INNER_SELECTED_FEATURES))
print("Threshold:", INNER_FINAL_THRESHOLD)

print(
    "Accuracy:",
    accuracy_score(
        y_test_inner_final,
        inner_test_pred_28
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_test_inner_final,
        inner_test_pred_28
    )
)

print(
    "F1:",
    f1_score(
        y_test_inner_final,
        inner_test_pred_28
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test_inner_final,
        inner_test_prob_28
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test_inner_final,
        inner_test_pred_28
    )
)

In [ ]:
# Per-bearing performance on the completely unseen test bearings

inner_final_test_results = test_inner_fault_aware_df[
    [
        "file",
        "bearing",
        "operating_condition",
        "measurement_number",
        "fault_class",
        "binary_label"
    ]
].copy()

inner_final_test_results["prediction"] = (
    inner_test_pred_28
)

inner_final_test_results["probability_inner"] = (
    inner_test_prob_28
)

final_inner_bearing_results = []

for bearing, group in (
    inner_final_test_results.groupby("bearing")
):

    final_inner_bearing_results.append({
        "bearing": bearing,
        "class": (
            "Healthy"
            if group["fault_class"].iloc[0] == 0
            else "Inner"
        ),
        "samples": len(group),
        "accuracy": accuracy_score(
            group["binary_label"],
            group["prediction"]
        ),
        "errors": (
            group["binary_label"] !=
            group["prediction"]
        ).sum(),
        "mean_probability": group[
            "probability_inner"
        ].mean()
    })

display(
    pd.DataFrame(
        final_inner_bearing_results
    )
    .sort_values("bearing")
    .round(4)
)

# 7. Integrated Healthy vs Damaged experiment

Map Healthy to 0 and Outer/Inner to 1. Both (class 3) is excluded. The code defines 21 training bearings, 4 validation bearings and 4 test bearings and checks that physical-bearing IDs do not overlap.

In [ ]:
# Step 8A
# Integrated Healthy vs Damaged experiment
#
# Healthy = 0
# Outer   = 1
# Inner   = 1
# Both    = excluded for now

df_integrated = df[
    df["fault_class"].isin([0, 1, 2])
].copy()

df_integrated["binary_label"] = (
    df_integrated["fault_class"] != 0
).astype(int)

TRAIN_BEARINGS_INTEGRATED = [
    "K001", "K002", "K003", "K004",
    "KA01", "KA04", "KA05", "KA06",
    "KA07", "KA09", "KA15", "KA16", "KA30",
    "KI01", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17"
]

VAL_BEARINGS_INTEGRATED = [
    "K005",
    "KA03",
    "KA08",
    "KI03"
]

TEST_BEARINGS_INTEGRATED = [
    "K006",
    "KA22",
    "KI18",
    "KI21"
]


def assign_integrated_split(bearing):

    if bearing in TRAIN_BEARINGS_INTEGRATED:
        return "train"

    elif bearing in VAL_BEARINGS_INTEGRATED:
        return "val"

    elif bearing in TEST_BEARINGS_INTEGRATED:
        return "test"

    else:
        return None


df_integrated["split"] = (
    df_integrated["bearing"]
    .map(assign_integrated_split)
)

print("Dataset shape:", df_integrated.shape)

print("\nClass distribution:")
print(
    df_integrated["binary_label"]
    .value_counts()
)

print("\nOriginal fault distribution:")
print(
    df_integrated["fault_class"]
    .value_counts()
)

print("\nSplit distribution:")
print(
    df_integrated["split"]
    .value_counts()
)

print("\nBearings by split:")
print(
    df_integrated
    .groupby("split")["bearing"]
    .nunique()
)

print("\nUnassigned rows:")
print(
    df_integrated["split"].isna().sum()
)

In [ ]:
# Step 8A-1
# Identify bearings that were not assigned to a split.

unassigned_df = df_integrated[
    df_integrated["split"].isna()
].copy()

print("Unassigned bearings:")

print(
    unassigned_df[
        ["bearing", "fault_class", "operating_condition"]
    ]
    .drop_duplicates()
    .sort_values(["bearing", "fault_class"])
    .to_string(index=False)
)

print("\nRows per unassigned bearing:")

print(
    unassigned_df["bearing"].value_counts()
)

print("\nAll bearings in the dataset:")

print(
    sorted(df_integrated["bearing"].unique())
)

In [ ]:
# Step 8B
# Prepare integrated Healthy vs Damaged datasets

train_integrated_df = df_integrated[
    df_integrated["split"] == "train"
].copy()

val_integrated_df = df_integrated[
    df_integrated["split"] == "val"
].copy()

test_integrated_df = df_integrated[
    df_integrated["split"] == "test"
].copy()


# Check dataset sizes
print("Train shape:", train_integrated_df.shape)
print("Validation shape:", val_integrated_df.shape)
print("Test shape:", test_integrated_df.shape)


# Check binary class distributions
print("\nTraining distribution:")
print(train_integrated_df["binary_label"].value_counts())

print("\nValidation distribution:")
print(val_integrated_df["binary_label"].value_counts())

print("\nTest distribution:")
print(test_integrated_df["binary_label"].value_counts())


# Check physical bearings
print("\nTraining bearings:")
print(sorted(train_integrated_df["bearing"].unique()))

print("\nValidation bearings:")
print(sorted(val_integrated_df["bearing"].unique()))

print("\nTest bearings:")
print(sorted(test_integrated_df["bearing"].unique()))


# Verify there is no bearing overlap
train_bearings = set(train_integrated_df["bearing"])
val_bearings = set(val_integrated_df["bearing"])
test_bearings = set(test_integrated_df["bearing"])

print("\nBearing overlap:")
print("Train & Validation:", train_bearings & val_bearings)
print("Train & Test:", train_bearings & test_bearings)
print("Validation & Test:", val_bearings & test_bearings)


# Final consistency check
assert len(train_integrated_df) == 1680
assert len(val_integrated_df) == 319
assert len(test_integrated_df) == 320

assert not (train_bearings & val_bearings)
assert not (train_bearings & test_bearings)
assert not (val_bearings & test_bearings)

print("\nIntegrated split verification passed.")

In [ ]:
# Step 8C
# Unified BPFO + BPFI feature extraction

def get_signal_and_rpm(mat_file):

    mat = sio.loadmat(
        mat_file,
        squeeze_me=False,
        struct_as_record=False
    )

    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    if len(data_keys) != 1:
        raise ValueError(f"Unexpected MATLAB structure: {mat_file}")

    data = mat[data_keys[0]][0, 0]

    # Extract vibration_1
    y_channels = np.asarray(data.Y).flatten()

    signal = None

    for channel in y_channels:

        name = channel.Name

        if isinstance(name, np.ndarray):
            name = str(name.flatten()[0])

        if name == "vibration_1":
            signal = np.asarray(
                channel.Data
            ).squeeze().astype(np.float64)
            break

    if signal is None:
        raise ValueError(f"vibration_1 not found: {mat_file}")

    # Extract measured rotational speed
    speed = extract_channel(data, "speed")
    rpm = float(np.mean(speed))

    return signal, rpm


def compute_integrated_features(df_subset, split_name):

    records = []

    for i, (_, row) in enumerate(df_subset.iterrows()):

        path = find_mat_file(row["file"])

        signal, rpm = get_signal_and_rpm(path)

        # BPFO features: 0-500 Hz
        bpfo_features = extract_bpfo_energy_features(
            signal,
            rpm,
            max_envelope_freq=500
        )

        bpfo_total = sum(
            bpfo_features[
                f"bpfo_{h}x_energy_ratio"
            ]
            for h in range(1, 6)
        )

        # BPFI features: 0-700 Hz
        bpfi_features = extract_bpfi_energy_features(
            signal,
            rpm,
            max_envelope_freq=700
        )

        combined_features = {
            **bpfo_features,
            "bpfo_1to5_total": bpfo_total,
            **bpfi_features
        }

        combined_features["file"] = row["file"]

        records.append(combined_features)

        if (i + 1) % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{i + 1}/{len(df_subset)}"
            )

    return pd.DataFrame(records)

In [ ]:
print("Processing integrated training data...")

integrated_train_features = compute_integrated_features(
    train_integrated_df,
    "Training"
)

print("\nProcessing integrated validation data...")

integrated_val_features = compute_integrated_features(
    val_integrated_df,
    "Validation"
)

print("\nProcessing integrated test data...")

integrated_test_features = compute_integrated_features(
    test_integrated_df,
    "Test"
)

print("\nFeature extraction completed.")

print("Training:", integrated_train_features.shape)
print("Validation:", integrated_val_features.shape)
print("Test:", integrated_test_features.shape)

In [ ]:
BPFO_INTEGRATED_FEATURES = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_1to5_total"
]

BPFI_INTEGRATED_FEATURES = [
    "bpfi_1x_energy_ratio",
    "bpfi_2x_energy_ratio",
    "bpfi_3x_energy_ratio",
    "bpfi_4x_energy_ratio",
    "bpfi_5x_energy_ratio",
    "bpfi_1to5_total"
]

UNIFIED_FAULT_FEATURES = (
    BPFO_INTEGRATED_FEATURES +
    BPFI_INTEGRATED_FEATURES
)

print("Original features:", len(FEATURES_22))
print("BPFO features:", len(BPFO_INTEGRATED_FEATURES))
print("BPFI features:", len(BPFI_INTEGRATED_FEATURES))
print("Unified feature count:",
      len(FEATURES_22 + UNIFIED_FAULT_FEATURES))

print("\nMissing values:")

print(
    integrated_train_features[
        UNIFIED_FAULT_FEATURES
    ].isna().sum()
)

In [ ]:
# Step 8D
# Merge original features and unified BPFO/BPFI features

UNIFIED_FEATURES = FEATURES_22 + UNIFIED_FAULT_FEATURES


def merge_integrated_features(base_df, feature_df, split_name):

    print(f"\nChecking {split_name}...")

    # Check duplicate filenames
    print("Duplicate feature filenames:",
          feature_df["file"].duplicated().sum())

    merged = base_df.merge(
        feature_df[["file"] + UNIFIED_FAULT_FEATURES],
        on="file",
        how="left",
        validate="one_to_one"
    )

    assert len(merged) == len(base_df), (
        f"Row count changed in {split_name}"
    )

    assert merged[UNIFIED_FAULT_FEATURES].isna().sum().sum() == 0, (
        f"Missing fault features in {split_name}"
    )

    print("Original rows:", len(base_df))
    print("Merged rows:", len(merged))
    print("Feature count:", len(UNIFIED_FEATURES))

    return merged


train_unified_df = merge_integrated_features(
    train_integrated_df,
    integrated_train_features,
    "Training"
)

val_unified_df = merge_integrated_features(
    val_integrated_df,
    integrated_val_features,
    "Validation"
)

test_unified_df = merge_integrated_features(
    test_integrated_df,
    integrated_test_features,
    "Test"
)

print("\nAll three datasets merged successfully.")

In [ ]:
# Step 8D-1
# Final integrity verification

for name, data in [
    ("Train", train_unified_df),
    ("Validation", val_unified_df),
    ("Test", test_unified_df)
]:

    print(f"\n{name}")
    print("-" * 35)

    print("Shape:", data.shape)
    print("Unique bearings:", data["bearing"].nunique())
    print("Unique files:", data["file"].nunique())

    print("\nClass distribution:")
    print(data["binary_label"].value_counts())

    print("\nMissing unified features:")
    print(data[UNIFIED_FAULT_FEATURES].isna().sum())


# Verify bearing separation
assert set(train_unified_df["bearing"]).isdisjoint(
    val_unified_df["bearing"]
)

assert set(train_unified_df["bearing"]).isdisjoint(
    test_unified_df["bearing"]
)

assert set(val_unified_df["bearing"]).isdisjoint(
    test_unified_df["bearing"]
)

print("\nUnified dataset integrity verification passed.")

# 8. Grouped cross-validation, feature comparisons and bearing diagnostics

Combine the 22 general features with six BPFO and six BPFI features. GroupKFold uses physical bearing ID as the grouping variable, and the scaler is fitted only on each fold's training data. Compare feature groups, models, ablations and per-bearing outcomes.

In [ ]:
# Step 8E-1
# Define integrated candidate feature representations

BPFO_HARMONICS = [
    f"bpfo_{i}x_energy_ratio"
    for i in range(1, 6)
]

BPFI_HARMONICS = [
    f"bpfi_{i}x_energy_ratio"
    for i in range(1, 6)
]

INTEGRATED_FEATURE_SETS = {

    "Original 22": FEATURES_22,

    "Original + BPFO total": (
        FEATURES_22 + ["bpfo_1to5_total"]
    ),

    "Original + BPFI total": (
        FEATURES_22 + ["bpfi_1to5_total"]
    ),

    "Original + BPFO+BPFI totals": (
        FEATURES_22 +
        ["bpfo_1to5_total", "bpfi_1to5_total"]
    ),

    "Original + individual harmonics": (
        FEATURES_22 +
        BPFO_HARMONICS +
        BPFI_HARMONICS
    ),

    "Original + all fault features": (
        FEATURES_22 +
        UNIFIED_FAULT_FEATURES
    )
}

print("Candidate representations:\n")

for name, features in INTEGRATED_FEATURE_SETS.items():
    print(f"{name}: {len(features)} features")


# ------------------------------------------------
# Prepare four bearing-grouped folds
# ------------------------------------------------

from sklearn.model_selection import GroupKFold

X_cv = train_unified_df[FEATURES_22]
y_cv = train_unified_df["binary_label"]
groups_cv = train_unified_df["bearing"]

integrated_gkf = GroupKFold(n_splits=4)

integrated_cv_splits = list(
    integrated_gkf.split(
        X_cv,
        y_cv,
        groups=groups_cv
    )
)


# ------------------------------------------------
# Inspect every fold
# ------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(
    integrated_cv_splits,
    start=1
):

    fold_train = train_unified_df.iloc[train_idx]
    fold_val = train_unified_df.iloc[val_idx]

    train_bearings = set(fold_train["bearing"])
    heldout_bearings = set(fold_val["bearing"])

    assert train_bearings.isdisjoint(heldout_bearings)

    assert set(fold_train["binary_label"].unique()) == {0, 1}
    assert set(fold_val["binary_label"].unique()) == {0, 1}

    print(f"\nFOLD {fold}")
    print("-" * 40)

    print("Training rows:", len(fold_train))
    print("Validation rows:", len(fold_val))

    print("Training classes:")
    print(fold_train["binary_label"].value_counts().to_dict())

    print("Held-out classes:")
    print(fold_val["binary_label"].value_counts().to_dict())

    print("\nHeld-out bearings:")

    print(
        fold_val[
            ["bearing", "fault_class"]
        ]
        .drop_duplicates()
        .sort_values("bearing")
        .to_string(index=False)
    )

print("\nFour-fold bearing separation verified.")

In [ ]:
# Step 8E-2
# Compare integrated feature representations using GroupKFold

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

integrated_cv_results = []

for representation, feature_list in INTEGRATED_FEATURE_SETS.items():

    print(f"\nEvaluating: {representation}")

    X = train_unified_df[feature_list]
    y = train_unified_df["binary_label"]

    for fold, (train_idx, val_idx) in enumerate(
        integrated_cv_splits,
        start=1
    ):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Scale using training fold only
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train_fold
        )

        X_val_scaled = scaler.transform(
            X_val_fold
        )

        # Same classifier for every experiment
        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(
            X_train_scaled,
            y_train_fold
        )

        # Predictions
        y_pred = model.predict(X_val_scaled)

        y_prob = model.predict_proba(
            X_val_scaled
        )[:, 1]

        integrated_cv_results.append({

            "representation": representation,
            "fold": fold,
            "features": len(feature_list),

            "accuracy": accuracy_score(
                y_val_fold,
                y_pred
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_val_fold,
                y_pred
            ),

            "f1": f1_score(
                y_val_fold,
                y_pred
            ),

            "roc_auc": roc_auc_score(
                y_val_fold,
                y_prob
            )
        })


integrated_cv_df = pd.DataFrame(
    integrated_cv_results
)

print("\nIndividual fold results:")

display(
    integrated_cv_df.round(4)
)

In [ ]:
# Step 8E-3
# Mean and standard deviation across the four folds

integrated_cv_summary = (
    integrated_cv_df
    .groupby(["representation", "features"])
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),

        mean_balanced_accuracy=(
            "balanced_accuracy", "mean"
        ),

        std_balanced_accuracy=(
            "balanced_accuracy", "std"
        ),

        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),

        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std")
    )
    .reset_index()
    .sort_values(
        "mean_roc_auc",
        ascending=False
    )
)

display(
    integrated_cv_summary.round(4)
)

In [ ]:
# Step 8F
# Diagnose performance across individual held-out bearings

DIAGNOSTIC_FEATURE_SETS = {
    "Original 22": FEATURES_22,
    "Original + all fault features": (
        INTEGRATED_FEATURE_SETS[
            "Original + all fault features"
        ]
    )
}

fold_diagnostic_results = []
bearing_diagnostic_results = []

for representation, features in DIAGNOSTIC_FEATURE_SETS.items():

    X = train_unified_df[features]
    y = train_unified_df["binary_label"]

    for fold, (train_idx, val_idx) in enumerate(
        integrated_cv_splits,
        start=1
    ):

        train_fold = train_unified_df.iloc[train_idx]
        val_fold = train_unified_df.iloc[val_idx]

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(X_train_scaled, y_train)

        predictions = model.predict(X_val_scaled)
        probabilities = model.predict_proba(X_val_scaled)[:, 1]

        fold_diagnostic_results.append({
            "representation": representation,
            "fold": fold,
            "heldout_bearings": ", ".join(
                sorted(val_fold["bearing"].unique())
            ),
            "accuracy": accuracy_score(y_val, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "roc_auc": roc_auc_score(y_val, probabilities)
        })

        # Individual bearing performance
        fold_predictions = val_fold[
            ["bearing", "fault_class", "binary_label"]
        ].copy()

        fold_predictions["prediction"] = predictions

        for bearing, group in fold_predictions.groupby("bearing"):

            bearing_diagnostic_results.append({
                "representation": representation,
                "fold": fold,
                "bearing": bearing,
                "fault_class": group["fault_class"].iloc[0],
                "samples": len(group),
                "accuracy": accuracy_score(
                    group["binary_label"],
                    group["prediction"]
                ),
                "errors": (
                    group["binary_label"] !=
                    group["prediction"]
                ).sum()
            })


fold_diagnostic_df = pd.DataFrame(
    fold_diagnostic_results
)

bearing_diagnostic_df = pd.DataFrame(
    bearing_diagnostic_results
)

print("FOLD-LEVEL RESULTS")

display(
    fold_diagnostic_df.round(4)
)

print("\nBEARING-LEVEL RESULTS")

display(
    bearing_diagnostic_df
    .sort_values(
        ["representation", "fold", "bearing"]
    )
    .round(4)
)

In [ ]:
# Step 8G
# Compare the four Healthy training bearings
# across all four operating conditions.

DIAGNOSTIC_FEATURES = [
    "rms",
    "std",
    "kurtosis",
    "skewness",
    "crest_factor",
    "dominant_frequency",
    "dominant_magnitude",
    "spectral_centroid",
    "spectral_bandwidth",
    "band_power_500_1000",
    "band_power_9000_10000",
    "bpfo_1to5_total",
    "bpfi_1to5_total"
]

# Healthy bearing profiles
healthy_profiles = (
    train_unified_df[
        train_unified_df["fault_class"] == 0
    ]
    .groupby(["bearing", "operating_condition"])[
        DIAGNOSTIC_FEATURES
    ]
    .median()
    .reset_index()
)

print("HEALTHY BEARING PROFILES")
display(healthy_profiles.round(4))


# Compare with Outer and Inner fault distributions
damaged_profiles = (
    train_unified_df[
        train_unified_df["fault_class"].isin([1, 2])
    ]
    .groupby(["fault_class", "operating_condition"])[
        DIAGNOSTIC_FEATURES
    ]
    .median()
    .reset_index()
)

print("\nDAMAGED BEARING REFERENCE PROFILES")
display(damaged_profiles.round(4))

In [ ]:
PROFILE_FEATURES = [
    "rms",
    "kurtosis",
    "spectral_centroid",
    "band_power_500_1000",
    "band_power_9000_10000",
    "bpfo_1to5_total",
    "bpfi_1to5_total"
]

bearing_level_profiles = (
    train_unified_df
    .groupby(["bearing", "fault_class"])[PROFILE_FEATURES]
    .median()
    .reset_index()
    .sort_values(["fault_class", "bearing"])
)

print("BEARING-LEVEL MEDIAN PROFILES")
display(bearing_level_profiles.round(4))

print("\nCLASS-LEVEL RANGES OF BEARING MEDIANS")

class_profile_ranges = (
    bearing_level_profiles
    .groupby("fault_class")[PROFILE_FEATURES]
    .agg(["median", "min", "max"])
)

display(class_profile_ranges.round(4))

In [ ]:
FAULT_FEATURES = [
    "bpfo_1to5_total",
    "bpfi_1to5_total"
]

bearing_fault_quantiles = (
    train_unified_df
    .groupby(["bearing", "fault_class"])[FAULT_FEATURES]
    .quantile([0.10, 0.50, 0.90])
    .unstack(level=-1)
)

bearing_fault_quantiles.columns = [
    f"{feature}_q{int(quantile * 100)}"
    for feature, quantile in bearing_fault_quantiles.columns
]

bearing_fault_quantiles = bearing_fault_quantiles.reset_index()

print("RECORDING-LEVEL FAULT FEATURE DISTRIBUTIONS")
display(bearing_fault_quantiles.round(5))

In [ ]:
condition_fault_quantiles = (
    train_unified_df
    .groupby(
        ["bearing", "fault_class", "operating_condition"]
    )[FAULT_FEATURES]
    .quantile([0.10, 0.50, 0.90])
    .unstack(level=-1)
)

condition_fault_quantiles.columns = [
    f"{feature}_q{int(quantile * 100)}"
    for feature, quantile in condition_fault_quantiles.columns
]

condition_fault_quantiles = (
    condition_fault_quantiles
    .reset_index()
    .sort_values(["operating_condition", "fault_class", "bearing"])
)

print("FAULT-FEATURE DISTRIBUTIONS BY BEARING AND OPERATING CONDITION")
display(condition_fault_quantiles.round(5))

In [ ]:
# First: calculate median fault features for each physical bearing
# under each operating condition.

condition_bearing_medians = (
    train_unified_df
    .groupby(["bearing", "fault_class", "operating_condition"])[FAULT_FEATURES]
    .median()
    .reset_index()
)

# Second: summarize the bearing-level medians by class and condition.

summary_aggregations = {
    "bearing_count": ("bearing", "nunique")
}

for feature in FAULT_FEATURES:
    summary_aggregations[f"{feature}_median"] = (feature, "median")
    summary_aggregations[f"{feature}_min"] = (feature, "min")
    summary_aggregations[f"{feature}_max"] = (feature, "max")

condition_class_summary = (
    condition_bearing_medians
    .groupby(["operating_condition", "fault_class"])
    .agg(**summary_aggregations)
    .reset_index()
)

condition_class_summary["fault_class"] = (
    condition_class_summary["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

print("FAULT-FEATURE SUMMARY BY OPERATING CONDITION")
display(condition_class_summary.round(5))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

# Work only with the integrated training dataset.
cv_data = train_unified_df.copy()

# Binary labels: Healthy = 0, Outer/Inner = 1.
y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

# New OR-like fault-energy feature.
cv_data["fault_energy_max"] = np.maximum(
    cv_data["bpfo_1to5_total"],
    cv_data["bpfi_1to5_total"]
)

# Compare different feature representations.
feature_sets = {
    "Original 22": FEATURES_22,

    "Original + BPFO/BPFI totals": (
        FEATURES_22 + [
            "bpfo_1to5_total",
            "bpfi_1to5_total"
        ]
    ),

    "Original + fault energy max": (
        FEATURES_22 + ["fault_energy_max"]
    ),

    "Original + totals + max": (
        FEATURES_22 + [
            "bpfo_1to5_total",
            "bpfi_1to5_total",
            "fault_energy_max"
        ]
    ),

    "Original + all fault features": (
        FEATURES_22 + UNIFIED_FAULT_FEATURES
    )
}

group_cv = GroupKFold(n_splits=4)

cv_results = []

for feature_name, features in feature_sets.items():

    X = cv_data[features].to_numpy()

    fold_number = 1

    for train_idx, val_idx in group_cv.split(X, y, groups):

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fit preprocessing only on the fold's training bearings.
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(X_train_scaled, y_train)

        probabilities = model.predict_proba(X_val_scaled)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        cv_results.append({
            "feature_set": feature_name,
            "fold": fold_number,
            "accuracy": accuracy_score(y_val, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "f1": f1_score(y_val, predictions, zero_division=0),
            "auc": roc_auc_score(y_val, probabilities)
        })

        fold_number += 1


# Display individual fold results.
cv_results_df = pd.DataFrame(cv_results)

print("INDIVIDUAL GROUP CV RESULTS")
display(cv_results_df.round(4))


# Calculate mean and standard deviation across folds.
cv_comparison = (
    cv_results_df
    .groupby("feature_set")
    .agg({
        "accuracy": ["mean", "std"],
        "balanced_accuracy": ["mean", "std"],
        "f1": ["mean", "std"],
        "auc": ["mean", "std"]
    })
)

print("\nFEATURE REPRESENTATION COMPARISON")
display(cv_comparison.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Use only integrated training data.
cv_data = train_unified_df.copy()

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[FEATURES_DIAGNOSTIC].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,   # Healthy
    1: 1,   # Outer
    2: 1    # Inner
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

group_cv = GroupKFold(n_splits=4)

prediction_records = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    probabilities = model.predict_proba(X_val_scaled)[:, 1]

    predictions = (probabilities >= 0.5).astype(int)

    for i, original_idx in enumerate(val_idx):

        prediction_records.append({
            "bearing": cv_data.iloc[original_idx]["bearing"],
            "fault_class": cv_data.iloc[original_idx]["fault_class"],
            "fold": fold,
            "actual_label": y_val[i],
            "predicted_probability": probabilities[i],
            "predicted_label": predictions[i]
        })


# Store all out-of-fold predictions.
integrated_oof_predictions = pd.DataFrame(prediction_records)


# Summarize predictions for each physical bearing.
bearing_probability_summary = (
    integrated_oof_predictions
    .groupby(["bearing", "fault_class", "fold"])
    .agg(
        recordings=("predicted_probability", "count"),
        probability_mean=("predicted_probability", "mean"),
        probability_median=("predicted_probability", "median"),
        probability_q10=(
            "predicted_probability",
            lambda x: x.quantile(0.10)
        ),
        probability_q90=(
            "predicted_probability",
            lambda x: x.quantile(0.90)
        ),
        predicted_damaged_pct=(
            "predicted_label",
            lambda x: 100 * (x == 1).mean()
        ),
        bearing_accuracy=(
            "predicted_label",
            lambda x: 100 * (
                x == integrated_oof_predictions.loc[
                    x.index, "actual_label"
                ]
            ).mean()
        )
    )
    .reset_index()
    .sort_values(["fault_class", "bearing"])
)

bearing_probability_summary["class_name"] = (
    bearing_probability_summary["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

print("OUT-OF-FOLD PREDICTION CONFIDENCE BY BEARING")

display(
    bearing_probability_summary[
        [
            "bearing",
            "class_name",
            "fold",
            "recordings",
            "probability_mean",
            "probability_median",
            "probability_q10",
            "probability_q90",
            "predicted_damaged_pct",
            "bearing_accuracy"
        ]
    ].round(4)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Only use integrated training data.
cv_data = train_unified_df.copy()

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[FEATURES_DIAGNOSTIC].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

DIAGNOSTIC_BEARINGS = [
    "K001", "K002", "K003", "K004",
    "KA05", "KI05"
]

group_cv = GroupKFold(n_splits=4)

contribution_results = []
bearing_prediction_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train = y[train_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # Each feature's contribution to the log-odds.
    contributions = X_val_scaled * model.coef_[0]

    # Find the bearings belonging to this validation fold.
    val_bearings = cv_data.iloc[val_idx]["bearing"].to_numpy()

    probabilities = model.predict_proba(X_val_scaled)[:, 1]

    for bearing in DIAGNOSTIC_BEARINGS:

        mask = val_bearings == bearing

        if not mask.any():
            continue

        # Average contribution of each feature for this bearing.
        mean_contributions = pd.Series(
            contributions[mask].mean(axis=0),
            index=FEATURES_DIAGNOSTIC
        )

        mean_logit = model.decision_function(
            X_val_scaled[mask]
        ).mean()

        mean_probability = probabilities[mask].mean()

        bearing_prediction_results.append({
            "bearing": bearing,
            "fold": fold,
            "mean_logit": mean_logit,
            "mean_damaged_probability": mean_probability
        })

        # Five strongest features pushing towards Damaged.
        top_positive = mean_contributions.nlargest(5)

        # Five strongest features pushing towards Healthy.
        top_negative = mean_contributions.nsmallest(5)

        for feature, value in top_positive.items():
            contribution_results.append({
                "bearing": bearing,
                "fold": fold,
                "feature": feature,
                "contribution": value,
                "direction": "Towards Damaged"
            })

        for feature, value in top_negative.items():
            contribution_results.append({
                "bearing": bearing,
                "fold": fold,
                "feature": feature,
                "contribution": value,
                "direction": "Towards Healthy"
            })


# Display bearing-level prediction summary.
prediction_summary = pd.DataFrame(
    bearing_prediction_results
)

print("BEARING PREDICTION SUMMARY")
display(prediction_summary.round(4))


# Display the strongest feature contributions.
contribution_summary = pd.DataFrame(
    contribution_results
)

print("\nTOP FEATURE CONTRIBUTIONS")
display(
    contribution_summary
    .sort_values(["bearing", "contribution"],
                 ascending=[True, False])
    .round(4)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cv_data = train_unified_df.copy()

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[FEATURES_DIAGNOSTIC].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

DIAGNOSTIC_BEARINGS = [
    "K001", "K002", "K003", "K004",
    "KA05", "KI05"
]

# Define feature families.
time_features = [
    "mean", "std", "min", "max",
    "kurtosis", "skewness", "crest_factor"
]

bpfo_features = [
    f for f in FEATURES_DIAGNOSTIC
    if f.startswith("bpfo_")
]

bpfi_features = [
    f for f in FEATURES_DIAGNOSTIC
    if f.startswith("bpfi_")
]

spectral_features = [
    f for f in FEATURES_DIAGNOSTIC
    if f not in time_features
    and f not in bpfo_features
    and f not in bpfi_features
]

feature_groups = {
    "Time/statistical": time_features,
    "General spectral": spectral_features,
    "BPFO": bpfo_features,
    "BPFI": bpfi_features
}

# Verify that all features are assigned exactly once.
assigned_features = [
    feature
    for feature_list in feature_groups.values()
    for feature in feature_list
]

assert len(assigned_features) == len(FEATURES_DIAGNOSTIC)
assert len(set(assigned_features)) == len(FEATURES_DIAGNOSTIC)

group_cv = GroupKFold(n_splits=4)

group_contribution_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train = y[train_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # Calculate contribution of each feature.
    contributions = X_val_scaled * model.coef_[0]

    val_bearings = (
        cv_data.iloc[val_idx]["bearing"].to_numpy()
    )

    probabilities = model.predict_proba(X_val_scaled)[:, 1]

    for bearing in DIAGNOSTIC_BEARINGS:

        mask = val_bearings == bearing

        if not mask.any():
            continue

        # Average contribution of each feature.
        mean_contributions = pd.Series(
            contributions[mask].mean(axis=0),
            index=FEATURES_DIAGNOSTIC
        )

        # Sum contributions within each feature family.
        result = {
            "bearing": bearing,
            "fold": fold,
            "mean_damaged_probability": probabilities[mask].mean()
        }

        for group_name, features in feature_groups.items():
            result[group_name] = mean_contributions[features].sum()

        result["intercept"] = model.intercept_[0]

        result["reconstructed_logit"] = (
            sum(
                result[group_name]
                for group_name in feature_groups
            ) + result["intercept"]
        )

        result["actual_mean_logit"] = (
            model.decision_function(X_val_scaled[mask]).mean()
        )

        group_contribution_results.append(result)


group_contribution_df = pd.DataFrame(
    group_contribution_results
)

print("FEATURE-FAMILY CONTRIBUTION ANALYSIS")

display(
    group_contribution_df.round(4)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Training data only.
cv_data = train_unified_df.copy()

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[FEATURES_DIAGNOSTIC].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

group_cv = GroupKFold(n_splits=4)

coefficient_records = []
fold_records = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train = X[train_idx]
    y_train = y[train_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # Store fold information.
    train_bearings = sorted(
        cv_data.iloc[train_idx]["bearing"].unique()
    )

    fold_records.append({
        "fold": fold,
        "training_bearings": ", ".join(train_bearings),
        "healthy_recordings": int((y_train == 0).sum()),
        "damaged_recordings": int((y_train == 1).sum()),
        "intercept": model.intercept_[0]
    })

    # Store coefficients for every feature.
    for feature, coefficient in zip(
        FEATURES_DIAGNOSTIC,
        model.coef_[0]
    ):

        coefficient_records.append({
            "fold": fold,
            "feature": feature,
            "coefficient": coefficient
        })


coefficient_df = pd.DataFrame(coefficient_records)

fold_summary = pd.DataFrame(fold_records)

print("FOLD COMPOSITION AND INTERCEPTS")
display(fold_summary)


# Organise coefficients into a feature-by-fold table.
coefficient_pivot = (
    coefficient_df
    .pivot(
        index="feature",
        columns="fold",
        values="coefficient"
    )
)

print("\nFEATURE COEFFICIENTS ACROSS FOUR FOLDS")
display(coefficient_pivot.round(4))


# Check coefficient direction and stability.
coefficient_stability = (
    coefficient_df
    .groupby("feature")
    .agg(
        mean_coefficient=("coefficient", "mean"),
        std_coefficient=("coefficient", "std"),
        positive_folds=(
            "coefficient",
            lambda x: (x > 0).sum()
        ),
        negative_folds=(
            "coefficient",
            lambda x: (x < 0).sum()
        ),
        minimum=("coefficient", "min"),
        maximum=("coefficient", "max")
    )
    .sort_values("std_coefficient", ascending=False)
)

print("\nCOEFFICIENT STABILITY")
display(coefficient_stability.round(4))

In [ ]:
import numpy as np
import pandas as pd

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

# Use training data only.
X_corr = train_unified_df[FEATURES_DIAGNOSTIC].copy()

# Pearson correlation between all 34 features.
correlation_matrix = X_corr.corr(method="pearson")

# Extract unique feature pairs.
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

correlation_pairs["absolute_correlation"] = (
    correlation_pairs["correlation"].abs()
)

correlation_pairs = correlation_pairs.sort_values(
    "absolute_correlation",
    ascending=False
)

print("TOP 25 CORRELATED FEATURE PAIRS")

display(
    correlation_pairs.head(25).round(4)
)


# Extract strongly correlated pairs.
high_correlation_pairs = correlation_pairs[
    correlation_pairs["absolute_correlation"] >= 0.85
]

print("\nFEATURE PAIRS WITH ABSOLUTE CORRELATION >= 0.85")

display(high_correlation_pairs.round(4))


# Specifically examine correlations involving fault features.
fault_related_pairs = correlation_pairs[
    correlation_pairs["feature_1"].str.startswith(("bpfo_", "bpfi_"))
    | correlation_pairs["feature_2"].str.startswith(("bpfo_", "bpfi_"))
]

print("\nTOP 20 CORRELATIONS INVOLVING FAULT FEATURES")

display(
    fault_related_pairs.head(20).round(4)
)

In [ ]:
import numpy as np
import pandas as pd

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

# Take the 25 strongest correlated pairs from the previous experiment.
top_correlated_pairs = correlation_pairs.head(25)[
    ["feature_1", "feature_2"]
].copy()

condition_correlations = []

# Calculate correlation separately for each operating condition.
for condition in sorted(
    train_unified_df["operating_condition"].unique()
):

    condition_data = train_unified_df[
        train_unified_df["operating_condition"] == condition
    ]

    corr_matrix = condition_data[
        FEATURES_DIAGNOSTIC
    ].corr(method="pearson")

    for _, pair in top_correlated_pairs.iterrows():

        feature_1 = pair["feature_1"]
        feature_2 = pair["feature_2"]

        correlation = corr_matrix.loc[
            feature_1, feature_2
        ]

        condition_correlations.append({
            "feature_1": feature_1,
            "feature_2": feature_2,
            "operating_condition": condition,
            "correlation": correlation
        })


condition_correlations_df = pd.DataFrame(
    condition_correlations
)

# Convert the four condition-specific correlations into columns.
condition_correlation_wide = (
    condition_correlations_df
    .pivot(
        index=["feature_1", "feature_2"],
        columns="operating_condition",
        values="correlation"
    )
    .reset_index()
)

condition_columns = sorted(
    train_unified_df["operating_condition"].unique()
)

# Calculate correlation stability across conditions.
condition_correlation_wide["mean_abs_correlation"] = (
    condition_correlation_wide[condition_columns]
    .abs()
    .mean(axis=1)
)

condition_correlation_wide["conditions_above_085"] = (
    condition_correlation_wide[condition_columns]
    .abs()
    .ge(0.85)
    .sum(axis=1)
)

condition_correlation_wide = (
    condition_correlation_wide
    .sort_values(
        "mean_abs_correlation",
        ascending=False
    )
)

print("TOP FEATURE PAIRS: CORRELATION BY OPERATING CONDITION")

display(condition_correlation_wide.round(4))

In [ ]:
import numpy as np
import pandas as pd

FEATURES_DIAGNOSTIC = FEATURES_22 + UNIFIED_FAULT_FEATURES

# Take the 25 strongest pairs from the overall training correlation.
top_pairs = correlation_pairs.head(25)[
    ["feature_1", "feature_2"]
].copy()

within_class_records = []

class_names = {
    0: "Healthy",
    1: "Outer",
    2: "Inner"
}

for fault_class, class_name in class_names.items():

    for condition in sorted(
        train_unified_df["operating_condition"].unique()
    ):

        subset = train_unified_df[
            (train_unified_df["fault_class"] == fault_class) &
            (train_unified_df["operating_condition"] == condition)
        ]

        corr_matrix = subset[FEATURES_DIAGNOSTIC].corr()

        for _, pair in top_pairs.iterrows():

            feature_1 = pair["feature_1"]
            feature_2 = pair["feature_2"]

            correlation = corr_matrix.loc[
                feature_1, feature_2
            ]

            within_class_records.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "class_name": class_name,
                "operating_condition": condition,
                "correlation": correlation
            })


within_class_df = pd.DataFrame(within_class_records)

# Create one row per feature pair.
within_class_wide = (
    within_class_df
    .pivot(
        index=["feature_1", "feature_2"],
        columns=["class_name", "operating_condition"],
        values="correlation"
    )
)

# Give the columns readable names.
within_class_wide.columns = [
    f"{class_name}_{condition}"
    for class_name, condition in within_class_wide.columns
]

within_class_wide = within_class_wide.reset_index()

correlation_group_columns = [
    column for column in within_class_wide.columns
    if column not in ["feature_1", "feature_2"]
]

# Summarise how consistently each pair is correlated
# within the individual fault classes and conditions.
within_class_wide["mean_abs_correlation"] = (
    within_class_wide[correlation_group_columns]
    .abs()
    .mean(axis=1)
)

within_class_wide["groups_above_085"] = (
    within_class_wide[correlation_group_columns]
    .abs()
    .ge(0.85)
    .sum(axis=1)
)

within_class_wide = within_class_wide.sort_values(
    "mean_abs_correlation",
    ascending=False
)

print("CORRELATION WITHIN EACH FAULT CLASS AND CONDITION")
display(within_class_wide.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Training data only.
cv_data = train_unified_df.copy()

# Complete 34-feature representation.
BASE_FEATURES = FEATURES_22 + UNIFIED_FAULT_FEATURES

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()


# Define controlled feature-removal experiments.
feature_sets = {
    "Baseline 34": BASE_FEATURES,

    "Remove band_power_0_500": [
        f for f in BASE_FEATURES
        if f != "band_power_0_500"
    ],

    "Remove band_power_500_1000": [
        f for f in BASE_FEATURES
        if f != "band_power_500_1000"
    ],

    "Remove band_power_8000_9000": [
        f for f in BASE_FEATURES
        if f != "band_power_8000_9000"
    ],

    "Remove std": [
        f for f in BASE_FEATURES
        if f != "std"
    ],

    "Compact A": [
        f for f in BASE_FEATURES
        if f not in [
            "band_power_0_500",
            "band_power_8000_9000"
        ]
    ],

    "Compact B": [
        f for f in BASE_FEATURES
        if f not in [
            "band_power_500_1000",
            "std"
        ]
    ]
}


group_cv = GroupKFold(n_splits=4)

ablation_results = []

for representation, features in feature_sets.items():

    X = cv_data[features].to_numpy()

    for fold, (train_idx, val_idx) in enumerate(
        group_cv.split(X, y, groups),
        start=1
    ):

        X_train = X[train_idx]
        X_val = X[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        # Scaling fitted only on fold-training data.
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(X_train_scaled, y_train)

        probabilities = model.predict_proba(
            X_val_scaled
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).astype(int)

        from sklearn.metrics import (
            accuracy_score,
            balanced_accuracy_score,
            f1_score,
            roc_auc_score
        )

        ablation_results.append({
            "representation": representation,
            "fold": fold,
            "number_of_features": len(features),
            "accuracy": accuracy_score(
                y_val, predictions
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "f1": f1_score(
                y_val, predictions,
                zero_division=0
            ),
            "auc": roc_auc_score(
                y_val, probabilities
            )
        })


ablation_df = pd.DataFrame(ablation_results)


# Individual fold results.
print("INDIVIDUAL GROUP CV RESULTS")

display(
    ablation_df.round(4)
)


# Mean and standard deviation across folds.
ablation_summary = (
    ablation_df
    .groupby("representation")
    .agg(
        number_of_features=("number_of_features", "first"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std")
    )
    .sort_values(
        "balanced_accuracy_mean",
        ascending=False
    )
)

print("\nFEATURE-REMOVAL COMPARISON")

display(
    ablation_summary.round(4)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

# Training data only.
cv_data = train_unified_df.copy()

FEATURES_MODEL_COMPARISON = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[FEATURES_MODEL_COMPARISON].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

assert not np.isnan(X).any()

group_cv = GroupKFold(n_splits=4)

model_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]

    y_train, y_val = y[train_idx], y[val_idx]

    # Scaling for Logistic Regression and SVM.
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ),

        "RBF SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )
    }

    for model_name, model in models.items():

        if model_name == "Random Forest":

            # Random Forest does not require standardisation.
            model.fit(X_train, y_train)

            scores = model.predict_proba(X_val)[:, 1]

            predictions = (scores >= 0.5).astype(int)

        elif model_name == "RBF SVM":

            model.fit(X_train_scaled, y_train)

            # SVM decision scores are used for AUC.
            scores = model.decision_function(X_val_scaled)

            predictions = (scores >= 0).astype(int)

        else:

            model.fit(X_train_scaled, y_train)

            scores = model.predict_proba(X_val_scaled)[:, 1]

            predictions = (scores >= 0.5).astype(int)

        model_results.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_val, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "f1": f1_score(
                y_val, predictions,
                zero_division=0
            ),
            "auc": roc_auc_score(y_val, scores)
        })


# Individual fold results.
model_results_df = pd.DataFrame(model_results)

print("MODEL COMPARISON: INDIVIDUAL FOLDS")

display(model_results_df.round(4))


# Mean and standard deviation.
model_comparison_summary = (
    model_results_df
    .groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std")
    )
    .sort_values("balanced_accuracy_mean", ascending=False)
)

print("\nOVERALL MODEL COMPARISON")

display(model_comparison_summary.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


# Training data only.
cv_data = train_unified_df.copy()

MODEL_FEATURES = FEATURES_22 + UNIFIED_FAULT_FEATURES

X = cv_data[MODEL_FEATURES].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

group_cv = GroupKFold(n_splits=4)

bearing_model_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train = y[train_idx]

    # Fit scaler using fold-training data only.
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ),

        "RBF SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )
    }

    val_bearings = cv_data.iloc[val_idx]["bearing"].to_numpy()
    val_classes = cv_data.iloc[val_idx]["fault_class"].to_numpy()

    for model_name, model in models.items():

        if model_name == "Random Forest":
            model.fit(X_train, y_train)
            predictions = model.predict(X_val)

        else:
            model.fit(X_train_scaled, y_train)
            predictions = model.predict(X_val_scaled)

        fold_predictions = pd.DataFrame({
            "bearing": val_bearings,
            "fault_class": val_classes,
            "actual_label": y[val_idx],
            "predicted_label": predictions
        })

        for bearing, bearing_data in fold_predictions.groupby("bearing"):

            actual_label = int(
                bearing_data["actual_label"].iloc[0]
            )

            predicted_damaged_pct = (
                100 * bearing_data["predicted_label"].mean()
            )

            bearing_accuracy = (
                100 * (
                    bearing_data["actual_label"]
                    == bearing_data["predicted_label"]
                ).mean()
            )

            bearing_model_results.append({
                "model": model_name,
                "fold": fold,
                "bearing": bearing,
                "fault_class": int(
                    bearing_data["fault_class"].iloc[0]
                ),
                "actual_label": actual_label,
                "predicted_damaged_pct": predicted_damaged_pct,
                "bearing_accuracy": bearing_accuracy
            })


bearing_model_df = pd.DataFrame(bearing_model_results)

bearing_model_df["class_name"] = (
    bearing_model_df["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)


# Display Healthy-bearing predictions separately.
print("HEALTHY BEARING PREDICTIONS")

healthy_model_comparison = (
    bearing_model_df[
        bearing_model_df["actual_label"] == 0
    ][
        [
            "model",
            "fold",
            "bearing",
            "predicted_damaged_pct",
            "bearing_accuracy"
        ]
    ]
    .sort_values(["bearing", "model"])
)

display(healthy_model_comparison.round(2))


# Display damaged-bearing predictions.
print("\nDAMAGED BEARING PREDICTIONS")

damaged_model_comparison = (
    bearing_model_df[
        bearing_model_df["actual_label"] == 1
    ][
        [
            "model",
            "fold",
            "bearing",
            "class_name",
            "predicted_damaged_pct",
            "bearing_accuracy"
        ]
    ]
    .sort_values(["bearing", "model"])
)

display(damaged_model_comparison.round(2))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

# Training data only.
cv_data = train_unified_df.copy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()


# Define feature families.
TIME_FEATURES = [
    "mean", "std", "min", "max",
    "kurtosis", "skewness", "crest_factor"
]

GENERAL_SPECTRAL_FEATURES = [
    f for f in FEATURES_22
    if f not in TIME_FEATURES
]


# Compare individual feature families and combinations.
feature_sets = {

    "Time/statistical only":
        TIME_FEATURES,

    "General spectral only":
        GENERAL_SPECTRAL_FEATURES,

    "BPFO only":
        BPFO_INTEGRATED_FEATURES,

    "BPFI only":
        BPFI_INTEGRATED_FEATURES,

    "BPFO + BPFI only":
        UNIFIED_FAULT_FEATURES,

    "Original 22":
        FEATURES_22,

    "Original 22 + BPFO":
        FEATURES_22 + BPFO_INTEGRATED_FEATURES,

    "Original 22 + BPFI":
        FEATURES_22 + BPFI_INTEGRATED_FEATURES,

    "Original 22 + all fault features":
        FEATURES_22 + UNIFIED_FAULT_FEATURES
}


group_cv = GroupKFold(n_splits=4)

family_results = []

for representation, features in feature_sets.items():

    X = cv_data[features].to_numpy()

    for fold, (train_idx, val_idx) in enumerate(
        group_cv.split(X, y, groups),
        start=1
    ):

        X_train, X_val = X[train_idx], X[val_idx]

        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

        model.fit(X_train_scaled, y_train)

        probabilities = model.predict_proba(
            X_val_scaled
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).astype(int)

        family_results.append({
            "representation": representation,
            "fold": fold,
            "number_of_features": len(features),
            "accuracy": accuracy_score(
                y_val, predictions
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
            "f1": f1_score(
                y_val, predictions,
                zero_division=0
            ),
            "auc": roc_auc_score(
                y_val, probabilities
            )
        })


family_results_df = pd.DataFrame(family_results)


# Individual fold results.
print("FEATURE FAMILY RESULTS BY FOLD")

display(
    family_results_df.round(4)
)


# Overall comparison.
family_summary = (
    family_results_df
    .groupby("representation")
    .agg(
        number_of_features=("number_of_features", "first"),
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std")
    )
    .sort_values(
        "balanced_accuracy_mean",
        ascending=False
    )
)

print("\nFEATURE FAMILY COMPARISON")

display(
    family_summary.round(4)
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Use integrated training dataset only.
cv_data = train_unified_df.copy()

FAULT_ONLY_FEATURES = UNIFIED_FAULT_FEATURES

X = cv_data[FAULT_ONLY_FEATURES].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

group_cv = GroupKFold(n_splits=4)

fault_only_predictions = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]

    y_train = y[train_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    probabilities = model.predict_proba(
        X_val_scaled
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    val_data = cv_data.iloc[val_idx]

    for i, (_, row) in enumerate(val_data.iterrows()):

        fault_only_predictions.append({
            "bearing": row["bearing"],
            "fault_class": row["fault_class"],
            "fold": fold,
            "actual_label": y[val_idx[i]],
            "predicted_probability": probabilities[i],
            "predicted_label": predictions[i]
        })


fault_only_oof_df = pd.DataFrame(
    fault_only_predictions
)


# Summarise results by physical bearing.
fault_only_bearing_summary = (
    fault_only_oof_df
    .groupby(["bearing", "fault_class", "fold"])
    .agg(
        recordings=("actual_label", "count"),

        mean_damaged_probability=(
            "predicted_probability", "mean"
        ),

        median_damaged_probability=(
            "predicted_probability", "median"
        ),

        predicted_damaged_pct=(
            "predicted_label",
            lambda x: 100 * (x == 1).mean()
        ),

        bearing_accuracy=(
            "predicted_label",
            lambda x: 100 * (
                x == fault_only_oof_df.loc[
                    x.index, "actual_label"
                ]
            ).mean()
        )
    )
    .reset_index()
    .sort_values(["fault_class", "bearing"])
)

fault_only_bearing_summary["class_name"] = (
    fault_only_bearing_summary["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

print("BEARING-LEVEL PREDICTIONS: BPFO + BPFI ONLY")

display(
    fault_only_bearing_summary[
        [
            "bearing",
            "class_name",
            "fold",
            "recordings",
            "mean_damaged_probability",
            "median_damaged_probability",
            "predicted_damaged_pct",
            "bearing_accuracy"
        ]
    ].round(4)
)

In [ ]:
import pandas as pd

# Original 34-feature Logistic Regression results.
original_results = (
    bearing_model_df[
        bearing_model_df["model"] == "Logistic Regression"
    ][
        [
            "bearing",
            "fault_class",
            "fold",
            "bearing_accuracy"
        ]
    ]
    .rename(columns={
        "bearing_accuracy": "accuracy_34_features"
    })
)

# New 12-feature results.
fault_results = (
    fault_only_bearing_summary[
        [
            "bearing",
            "fault_class",
            "fold",
            "bearing_accuracy"
        ]
    ]
    .rename(columns={
        "bearing_accuracy": "accuracy_12_features"
    })
)

# Merge using the physical bearing.
bearing_comparison = original_results.merge(
    fault_results,
    on=["bearing", "fault_class", "fold"],
    how="inner",
    validate="one_to_one"
)

# Calculate improvement.
bearing_comparison["improvement_pp"] = (
    bearing_comparison["accuracy_12_features"]
    - bearing_comparison["accuracy_34_features"]
)

bearing_comparison["class_name"] = (
    bearing_comparison["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

bearing_comparison = bearing_comparison.sort_values(
    ["fault_class", "bearing"]
)

print("BEARING-LEVEL PERFORMANCE COMPARISON")

display(
    bearing_comparison[
        [
            "bearing",
            "class_name",
            "fold",
            "accuracy_34_features",
            "accuracy_12_features",
            "improvement_pp"
        ]
    ].round(2)
)


# Summarise improvements separately for each class.
class_improvement = (
    bearing_comparison
    .groupby("class_name")
    .agg(
        mean_accuracy_34=("accuracy_34_features", "mean"),
        mean_accuracy_12=("accuracy_12_features", "mean"),
        mean_improvement_pp=("improvement_pp", "mean"),
        improved_bearings=(
            "improvement_pp",
            lambda x: (x > 0).sum()
        ),
        worsened_bearings=(
            "improvement_pp",
            lambda x: (x < 0).sum()
        )
    )
)

print("\nIMPROVEMENT BY FAULT CLASS")

display(class_improvement.round(2))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Training data only.
cv_data = train_unified_df.copy()

X = cv_data[UNIFIED_FAULT_FEATURES].to_numpy()

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

group_cv = GroupKFold(n_splits=4)

condition_predictions = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X, y, groups),
    start=1
):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train = y[train_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    probabilities = model.predict_proba(X_val_scaled)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    val_data = cv_data.iloc[val_idx]

    for i, (_, row) in enumerate(val_data.iterrows()):

        condition_predictions.append({
            "bearing": row["bearing"],
            "fault_class": int(row["fault_class"]),
            "operating_condition": row["operating_condition"],
            "fold": fold,
            "actual_label": int(y[val_idx[i]]),
            "predicted_probability": probabilities[i],
            "predicted_label": predictions[i]
        })


fault_condition_oof_df = pd.DataFrame(condition_predictions)

fault_condition_oof_df["correct"] = (
    fault_condition_oof_df["actual_label"]
    == fault_condition_oof_df["predicted_label"]
)

fault_condition_oof_df["class_name"] = (
    fault_condition_oof_df["fault_class"].map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)


# Bearing-level performance under each operating condition.
condition_bearing_summary = (
    fault_condition_oof_df
    .groupby([
        "bearing",
        "class_name",
        "fold",
        "operating_condition"
    ])
    .agg(
        recordings=("actual_label", "count"),
        mean_damaged_probability=(
            "predicted_probability", "mean"
        ),
        predicted_damaged_pct=(
            "predicted_label",
            lambda x: 100 * x.mean()
        ),
        accuracy=(
            "correct",
            lambda x: 100 * x.mean()
        )
    )
    .reset_index()
)

# Focus on Healthy bearings and previously difficult damaged bearings.
FOCUS_BEARINGS = [
    "K001", "K002", "K003", "K004",
    "KA15", "KA30",
    "KI05", "KI07", "KI17"
]

print("BEARING PERFORMANCE BY OPERATING CONDITION")

display(
    condition_bearing_summary[
        condition_bearing_summary["bearing"].isin(FOCUS_BEARINGS)
    ]
    .sort_values(["class_name", "bearing", "operating_condition"])
    .round(4)
)


# Overall class-level comparison for each operating condition.
condition_class_summary = (
    condition_bearing_summary
    .groupby(["class_name", "operating_condition"])
    .agg(
        bearing_count=("bearing", "nunique"),
        mean_bearing_accuracy=("accuracy", "mean"),
        mean_predicted_damaged_pct=("predicted_damaged_pct", "mean"),
        mean_damaged_probability=("mean_damaged_probability", "mean")
    )
    .reset_index()
)

print("\nCLASS PERFORMANCE BY OPERATING CONDITION")

display(condition_class_summary.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold

# Training data only.
cv_data = train_unified_df.copy()

FAULT_FEATURES = UNIFIED_FAULT_FEATURES

y = cv_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

groups = cv_data["bearing"].to_numpy()

FOCUS_BEARINGS = [
    "K002", "K004",
    "KA15",
    "KI05", "KI07", "KI17"
]

class_names = {
    0: "Healthy",
    1: "Outer",
    2: "Inner"
}

group_cv = GroupKFold(n_splits=4)

reference_comparison = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(
        cv_data[FAULT_FEATURES],
        y,
        groups
    ),
    start=1
):

    fold_train = cv_data.iloc[train_idx].copy()
    fold_val = cv_data.iloc[val_idx].copy()

    # Reference medians: first aggregate recordings per physical
    # bearing, then aggregate across physical bearings.
    train_bearing_medians = (
        fold_train
        .groupby([
            "bearing",
            "fault_class",
            "operating_condition"
        ])[FAULT_FEATURES]
        .median()
        .reset_index()
    )

    reference_profiles = (
        train_bearing_medians
        .groupby([
            "fault_class",
            "operating_condition"
        ])[FAULT_FEATURES]
        .median()
    )

    # Inspect selected held-out bearings.
    for bearing in FOCUS_BEARINGS:

        target = fold_val[
            fold_val["bearing"] == bearing
        ]

        if target.empty:
            continue

        target_profiles = (
            target
            .groupby([
                "fault_class",
                "operating_condition"
            ])[FAULT_FEATURES]
            .median()
            .reset_index()
        )

        actual_class = int(
            target_profiles["fault_class"].iloc[0]
        )

        for _, row in target_profiles.iterrows():

            condition = row["operating_condition"]

            result = {
                "bearing": bearing,
                "fold": fold,
                "actual_class": class_names[actual_class],
                "operating_condition": condition,
                "target_bpfo": row["bpfo_1to5_total"],
                "target_bpfi": row["bpfi_1to5_total"]
            }

            # Obtain reference profiles for all three classes.
            for class_id, class_name in class_names.items():

                try:
                    ref = reference_profiles.loc[
                        (class_id, condition)
                    ]

                    result[f"{class_name}_bpfo"] = (
                        ref["bpfo_1to5_total"]
                    )

                    result[f"{class_name}_bpfi"] = (
                        ref["bpfi_1to5_total"]
                    )

                except KeyError:
                    result[f"{class_name}_bpfo"] = np.nan
                    result[f"{class_name}_bpfi"] = np.nan

            reference_comparison.append(result)


reference_comparison_df = pd.DataFrame(
    reference_comparison
)

reference_comparison_df = (
    reference_comparison_df
    .sort_values(["bearing", "operating_condition"])
)

print("FAULT-FREQUENCY REFERENCE COMPARISON")

display(
    reference_comparison_df.round(5)
)

# 9. Validation, final test and exported feature tables

Compare the original 34-feature representation with BPFO/BPFI-only features, report validation and final-test metrics, inspect per-bearing predictions, and export development/test tables for Notebook 09.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Select feature representations.
validation_feature_sets = {
    "Original 34 features": (
        FEATURES_22 + UNIFIED_FAULT_FEATURES
    ),

    "BPFO + BPFI only": UNIFIED_FAULT_FEATURES
}

# Use integrated train and validation datasets.
train_data = train_unified_df.copy()
val_data = val_unified_df.copy()

# Binary labels: Healthy = 0, Outer/Inner = 1.
y_train = train_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

y_val = val_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

assert not np.isnan(y_train).any()
assert not np.isnan(y_val).any()

validation_metrics = []
validation_predictions = []

for representation, features in validation_feature_sets.items():

    X_train = train_data[features].to_numpy()
    X_val = val_data[features].to_numpy()

    # Fit scaler only on training data.
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    probabilities = model.predict_proba(
        X_val_scaled
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # Aggregate validation metrics.
    validation_metrics.append({
        "representation": representation,
        "features": len(features),
        "accuracy": accuracy_score(y_val, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, predictions
        ),
        "f1": f1_score(
            y_val, predictions,
            zero_division=0
        ),
        "auc": roc_auc_score(y_val, probabilities)
    })

    # Store predictions for each validation recording.
    for i, (_, row) in enumerate(val_data.iterrows()):

        validation_predictions.append({
            "representation": representation,
            "bearing": row["bearing"],
            "fault_class": int(row["fault_class"]),
            "operating_condition": row["operating_condition"],
            "actual_label": y_val[i],
            "predicted_label": predictions[i],
            "damaged_probability": probabilities[i]
        })

    print(f"\n{representation}")
    print("Confusion matrix:")
    print(confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ))

    print("\nClassification report:")
    print(classification_report(
        y_val,
        predictions,
        target_names=["Healthy", "Damaged"],
        zero_division=0
    ))


# Display aggregate comparison.
validation_metrics_df = pd.DataFrame(validation_metrics)

print("\nVALIDATION METRIC COMPARISON")

display(validation_metrics_df.round(4))


# Display per-bearing validation results.
validation_predictions_df = pd.DataFrame(
    validation_predictions
)

validation_bearing_summary = (
    validation_predictions_df
    .groupby(["representation", "bearing", "fault_class"])
    .agg(
        recordings=("actual_label", "count"),
        mean_damaged_probability=("damaged_probability", "mean"),
        predicted_damaged_pct=(
            "predicted_label",
            lambda x: 100 * x.mean()
        ),
        accuracy=(
            "predicted_label",
            lambda x: 100 * (
                x == validation_predictions_df.loc[
                    x.index, "actual_label"
                ]
            ).mean()
        )
    )
    .reset_index()
)

validation_bearing_summary["class_name"] = (
    validation_bearing_summary["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

print("\nVALIDATION PERFORMANCE BY PHYSICAL BEARING")

display(validation_bearing_summary.round(4))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Combine training and validation after model development.
development_data = pd.concat(
    [train_unified_df, val_unified_df],
    ignore_index=True
)

# Integrated test data remains separate.
test_data = test_unified_df.copy()

# Two locked candidate representations.
final_feature_sets = {
    "Original 34 features": (
        FEATURES_22 + UNIFIED_FAULT_FEATURES
    ),

    "BPFO + BPFI only": UNIFIED_FAULT_FEATURES
}

y_development = development_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

y_test = test_data["fault_class"].map({
    0: 0,
    1: 1,
    2: 1
}).to_numpy()

final_test_results = []
final_test_predictions = []

for representation, features in final_feature_sets.items():

    X_development = development_data[features].to_numpy()
    X_test = test_data[features].to_numpy()

    # Fit scaler using development data only.
    scaler = StandardScaler()

    X_development_scaled = scaler.fit_transform(
        X_development
    )

    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    model.fit(X_development_scaled, y_development)

    probabilities = model.predict_proba(
        X_test_scaled
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # Calculate test metrics.
    final_test_results.append({
        "representation": representation,
        "features": len(features),
        "accuracy": accuracy_score(
            y_test, predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "f1": f1_score(
            y_test, predictions,
            zero_division=0
        ),
        "auc": roc_auc_score(
            y_test, probabilities
        )
    })

    print(f"\n{'='*55}")
    print(representation)
    print(f"{'='*55}")

    print("\nConfusion matrix:")
    print(confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        predictions,
        target_names=["Healthy", "Damaged"],
        zero_division=0
    ))

    # Preserve individual test predictions.
    for i, (_, row) in enumerate(test_data.iterrows()):

        final_test_predictions.append({
            "representation": representation,
            "bearing": row["bearing"],
            "fault_class": int(row["fault_class"]),
            "actual_label": y_test[i],
            "predicted_label": predictions[i],
            "damaged_probability": probabilities[i]
        })


# Overall comparison.
final_test_metrics_df = pd.DataFrame(
    final_test_results
)

print("\nFINAL INTEGRATED TEST COMPARISON")

display(final_test_metrics_df.round(4))


# Individual bearing performance.
final_test_predictions_df = pd.DataFrame(
    final_test_predictions
)

final_test_bearing_summary = (
    final_test_predictions_df
    .groupby([
        "representation",
        "bearing",
        "fault_class"
    ])
    .agg(
        recordings=("actual_label", "count"),
        mean_damaged_probability=(
            "damaged_probability", "mean"
        ),
        predicted_damaged_pct=(
            "predicted_label",
            lambda x: 100 * x.mean()
        ),
        accuracy=(
            "predicted_label",
            lambda x: 100 * (
                x == final_test_predictions_df.loc[
                    x.index, "actual_label"
                ]
            ).mean()
        )
    )
    .reset_index()
)

final_test_bearing_summary["class_name"] = (
    final_test_bearing_summary["fault_class"]
    .map({
        0: "Healthy",
        1: "Outer",
        2: "Inner"
    })
)

print("\nTEST PERFORMANCE BY PHYSICAL BEARING")

display(final_test_bearing_summary.round(4))

In [ ]:
from pathlib import Path
import pandas as pd

# Explicit project root.
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name.lower() == "notebooks"
    else Path.cwd()
)

assert PROJECT_ROOT.exists(), "Project directory not found!"

# Correct export directory.
BASELINE_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "baseline"
)

BASELINE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Combine training and validation data.
development_data = pd.concat(
    [train_unified_df, val_unified_df],
    ignore_index=True
)

# Keep test data separate.
test_data = test_unified_df.copy()

# Export both datasets.
development_path = (
    BASELINE_DATA_DIR / "paderborn_development_features.csv"
)

test_path = (
    BASELINE_DATA_DIR / "paderborn_integrated_test_features.csv"
)

development_data.to_csv(development_path, index=False)
test_data.to_csv(test_path, index=False)

# Verify the exports.
assert development_path.exists()
assert test_path.exists()

assert development_data["bearing"].nunique() == 25
assert test_data["bearing"].nunique() == 4

assert set(development_data["bearing"]).isdisjoint(
    set(test_data["bearing"])
)

print("DATA EXPORT VERIFIED")

print("Development records:", len(development_data))
print("Development bearings:", development_data["bearing"].nunique())

print("Test records:", len(test_data))
print("Test bearings:", test_data["bearing"].nunique())

print("\nDevelopment class distribution:")
print(
    development_data["fault_class"]
    .value_counts()
    .sort_index()
)

print("\nCorrect export locations:")
print(development_path)
print(test_path)

## Results from the original saved run

The original output for the integrated Healthy-vs-Damaged experiment reported:

| Representation | Features | Accuracy | Balanced accuracy | F1 | ROC AUC |
|---|---:|---:|---:|---:|---:|
| Original 34 features | 34 | 99.06% | 99.38% | 99.37% | 1.0000 |
| BPFO + BPFI only | 12 | 78.75% | 85.83% | 83.50% | 0.9894 |

For the original 34-feature model, the saved confusion matrix (Healthy first, Damaged second) was `[[80, 0], [3, 237]]`. The evaluation set contained 80 Healthy and 240 Damaged recordings.

The notebook's final export cell reports 1,999 development recordings from 25 physical bearings and 320 integrated-test recordings from four physical bearings. Notebook 09 uses the resulting baseline feature tables to package the inference model.

**Interpretation:** the 99.06% metric belongs to this integrated binary experiment and its selected test subset. The four-bearing test subset was inspected during iterative experimentation; it is not a pristine untouched external test or 320 independent bearings. The integrated task excludes Both.